
# DNTC strict OCR/PDF/NLP pipeline for Kaggle

Notebook này thay pipeline cũ bằng pipeline nghiêm ngặt hơn cho bộ scan **Đại Nam Nhất Thống Chí**:

- tự tải PDF từ Google Drive, hoặc đọc từ `/kaggle/input`;
- phân loại trang trước khi lấy text;
- chọn nguồn text theo từng trang: PDF text layer, Tesseract OCR, hoặc drop/audit;
- không đưa dòng/trang OCR thấp điểm vào final;
- sửa OCR tiếng Việt/domain theo luật có log;
- reflow paragraph, tách sentence-only;
- xuất final và audit đầy đủ vào `/kaggle/working/dntc_auto`.

Ưu tiên của notebook này là **precision hơn recall**: dòng nào OCR không chắc sẽ bị drop vào audit, không đưa vào final.


In [1]:

# ============================================================
# 0. CONFIG - chỉnh ở đây rồi Run All
# ============================================================
from pathlib import Path
import os

# Google Drive folder/file URLs. Có thể thay bằng Drive folder của bạn.
DRIVE_URLS = [
    "https://drive.google.com/drive/folders/1QZzyaozPRLcm5Y2nmUUbigyVFnsX_tkW",
]

# Nếu không muốn download Drive, để DRIVE_URLS = [] và notebook đọc từ /kaggle/input.
ALLOW_KAGGLE_INPUT_FALLBACK = True
LOCAL_INPUT_DIRS = [Path("/kaggle/input"), Path("/mnt/data"), Path(".")]

# Output đúng yêu cầu.
OUTPUT_DIR = Path("/kaggle/working/dntc_auto") if Path("/kaggle/working").exists() else Path("./dntc_auto")
RAW_DIR = OUTPUT_DIR / "raw_drive"
FINAL_DIR = OUTPUT_DIR / "final"
TEXT_DIR = FINAL_DIR / "texts"
AUDIT_DIR = OUTPUT_DIR / "audit"
CACHE_DIR = OUTPUT_DIR / "cache"
PKG_DIR = OUTPUT_DIR / "packages"

# Run modes.
FAST_TEST_MODE = False           # True: chạy thử nhanh một số trang.
FAST_TEST_MAX_PDFS = 2
FAST_TEST_MAX_PAGES_TOTAL = 50
FULL_RUN_MODE = not FAST_TEST_MODE
PIPELINE_VERSION = "dntc_auto_v7_no_han_text_layer"
MAX_PAGES_PER_PDF = None         # None = all pages; FAST_TEST_MODE sẽ override bằng tổng 50 pages.

# Required page filtering config.
CONTENT_START_MODE = "auto"      # "auto" hoặc "none".
KEEP_TITLE_PAGES = False
DROP_LIBRARY_PAGES = True
DROP_BLANK_PAGES = True
DROP_PATTERNED_PAGES = True
DROP_LOW_CONF_OCR_PAGES = True
DROP_FRONT_MATTER_BEFORE_CONTENT = True
DROP_BACKMATTER_PAGES = True          # drop publisher ads, print info, table-of-contents/end matter pages
STRICT_SENTENCE_ONLY = True         # sentence csv should not contain line fragments

# OCR controls. Không dùng PaddleOCR mặc định vì dependency nặng/dễ lỗi trên Kaggle.
ENABLE_TESSERACT = True
OCR_LANG = "vie+eng"
PAGE_OCR_DPI = 260
FAST_FRONTMATTER_OCR_DPI = 150
LINE_REOCR_ZOOM = 4.0
LINE_REOCR_PAD_PT = 7.0
MAX_LINE_REOCR_PER_PDF = 250

# Quality thresholds. Tăng nếu muốn ít final hơn nhưng sạch hơn.
MIN_SELECTED_PAGE_QUALITY = 43.0
MIN_OCR_PAGE_QUALITY = 46.0
MIN_TEXT_LAYER_GOOD_QUALITY = 55.0
MIN_KEEP_LINE_QUALITY = 38.0
MIN_KEEP_SENTENCE_QUALITY = 38.0
MIN_OCR_CONF_LINE = 35.0
MAX_LIBRARY_NOISE_SCORE = 0.18
MAX_WEIRD_CHAR_RATIO = 0.065
MIN_VIET_RATIO_FOR_LONG_TEXT = 0.10

# Content detection.
AUTO_CONTENT_SCAN_PAGES = 35
MIN_CONTENT_WORDS_ON_START_PAGE = 35
MIN_CONTENT_LINES_ON_START_PAGE = 4

# Exports.
ZIP_OUTPUT = True
WRITE_DEBUG_PAGE_THUMBNAILS = False
DEBUG_THUMBNAIL_MAX = 80

for d in [OUTPUT_DIR, RAW_DIR, FINAL_DIR, TEXT_DIR, AUDIT_DIR, CACHE_DIR, PKG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("OUTPUT_DIR:", OUTPUT_DIR)
print("FAST_TEST_MODE:", FAST_TEST_MODE)
print("FULL_RUN_MODE:", FULL_RUN_MODE)



OUTPUT_DIR: /kaggle/working/dntc_auto
FAST_TEST_MODE: False
FULL_RUN_MODE: True


In [2]:

# ============================================================
# 1. Install dependencies - Kaggle Run All friendly
# ============================================================
import sys, subprocess, shutil, importlib.util


def pip_install(*pkgs):
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *pkgs]
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=False)

required_modules = {
    "fitz": ["pymupdf"],
    "pandas": ["pandas"],
    "numpy": ["numpy"],
    "PIL": ["pillow"],
    "tqdm": ["tqdm"],
    "gdown": ["gdown"],
    "pytesseract": ["pytesseract"],
}
for mod, pkgs in required_modules.items():
    if importlib.util.find_spec(mod) is None:
        pip_install(*pkgs)

# Tesseract binary + Vietnamese and English traineddata.
if ENABLE_TESSERACT:
    missing_binary = shutil.which("tesseract") is None
    if missing_binary:
        print("Installing tesseract system packages...")
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "tesseract-ocr", "tesseract-ocr-vie", "tesseract-ocr-eng"], check=False)
    else:
        # Ensure vie/eng packages exist. Safe if already installed.
        try:
            langs = subprocess.check_output(["tesseract", "--list-langs"], text=True, stderr=subprocess.STDOUT)
        except Exception:
            langs = ""
        if "vie" not in langs or "eng" not in langs:
            print("Installing tesseract Vietnamese/English traineddata...")
            subprocess.run(["apt-get", "update", "-qq"], check=False)
            subprocess.run(["apt-get", "install", "-y", "tesseract-ocr-vie", "tesseract-ocr-eng"], check=False)

print("tesseract:", shutil.which("tesseract"))


Installing tesseract Vietnamese/English traineddata...


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr-eng is already the newest version (1:4.00~git30-7274cfa-1.1).
tesseract-ocr-eng set to manually installed.
The following NEW packages will be installed:
  tesseract-ocr-vie
0 upgraded, 1 newly installed, 0 to remove and 99 not upgraded.
Need to get 417 kB of archives.
After this operation, 546 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-vie all 1:4.00~git30-7274cfa-1.1 [417 kB]
Fetched 417 kB in 0s (1,914 kB/s)
Selecting previously unselected package tesseract-ocr-vie.
(Reading database ... 121026 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-vie_1%3a4.00~git30-7274cfa-1.1_all.deb ...
Unpacking tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
Setting up tesseract-ocr-vie (1:4.00~git30-7274cfa-1.1) ...
tesseract: /usr/bin/tesseract


In [3]:

# ============================================================
# 2. Imports and shared constants
# ============================================================
import os, re, math, json, time, zipfile, hashlib, shutil, subprocess, sys, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass

import numpy as np
import pandas as pd
import fitz
from PIL import Image, ImageOps, ImageEnhance, ImageFilter
from tqdm.auto import tqdm
import gdown

try:
    import pytesseract
    from pytesseract import Output
except Exception as e:
    pytesseract = None
    Output = None
    print("pytesseract not available:", repr(e))

VIET_CHARS = set(
    "ăâđêôơưáàảãạắằẳẵặấầẩẫậéèẻẽẹếềểễệíìỉĩị"
    "óòỏõọốồổỗộớờởỡợúùủũụứừửữựýỳỷỹỵ"
    "ĂÂĐÊÔƠƯÁÀẢÃẠẮẰẲẴẶẤẦẨẪẬÉÈẺẼẸẾỀỂỄỆÍÌỈĨỊ"
    "ÓÒỎÕỌỐỒỔỖỘỚỜỞỠỢÚÙỦŨỤỨỪỬỮỰÝỲỶỸỴ"
)
VIET_VOWELS = set("aeiouyAEIOUYàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵăâêôơưĂÂÊÔƠƯ")
LETTERS_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]")
WORDS_RE = re.compile(r"[A-Za-zÀ-ỹĐđ]+")
DIGIT_RE = re.compile(r"\d")
CONTROL_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\ufffe\uffff\u0001]")
CJK_RE = re.compile(r"[\u3400-\u9fff]")
NON_VIET_SCRIPT_RE = re.compile(r"[Α-ωА-яЁё]")
WEIRD_RE = re.compile(r"[^\w\sÀ-ỹ.,;:!?(){}\[\]\"'“”‘’/\-–—%°+&<>«»·•*]", re.UNICODE)

COMMON_VI_WORDS = set("""
của và là có không trong ngoài năm đời phủ huyện tỉnh châu xã thôn làng tổng phường sách dân người nước ta
sông núi biển cửa đông tây nam bắc phía giáp cách dặm linh thành đặt đổi thuộc đời nhà triều vua quan quân
quyền quyển đại nam nhất thống chí kinh sư phần dã dựng đặt diễn cách hình thế khí hậu phong tục thành trì
trường học hộ khẩu thuế ruộng núi sông cổ tích từ miếu đền chùa miếu đàn lăng mộ đồn lũy cửa biển cầu đường
minh mệnh gia long tự đức thiệu trị đồng khánh duy tân hiến tông duệ tông thế tổ thánh tông cao hoàng đế
chiêm thành cao mên xiêm la thanh nghệ an thanh hóa quảng bình quảng trị quảng nam quảng ngãi bình định
phú yên khánh hòa bình thuận hà tiên an giang biên hòa gia định định tường vĩnh long hà nội hải dương
""".split())

COMMON_UNACCENTED_VI_WORDS = set("""
cua va la co khong trong ngoai nam doi phu huyen tinh chau xa thon lang tong phuong sach dan nguoi nuoc ta
song nui bien cua dong tay nam bac phia giap cach dam linh thanh dat doi thuoc nha trieu vua quan quyen quyen
dai nam nhat thong chi kinh su phan da dung dat dien cach hinh the khi hau phong tuc thanh tri truong hoc ho khau
thue ruong co tich tu mieu den chua minh menh gia long tu duc thieu tri dong khanh duy tan hien tong due tong
the to thanh tong chiem thanh cao men xiem la quang binh quang tri quang nam quang ngai binh dinh phu yen
khanh hoa binh thuan ha tien an giang bien hoa gia dinh dinh tuong vinh long ha noi hai duong
""".split())

DOMAIN_HEADINGS = [
    "ĐẠI NAM NHẤT THỐNG CHÍ", "LỜI NÓI ĐẦU", "BÀI TỰ", "PHÀM LỆ", "DỰNG ĐẶT VÀ DIÊN CÁCH", "PHẦN DÃ",
    "HÌNH THẾ", "KHÍ HẬU", "PHONG TỤC", "THÀNH TRÌ", "TRƯỜNG HỌC", "HỘ KHẨU", "THUẾ RUỘNG", "NÚI SÔNG",
    "SÔNG NGÒI", "CỔ TÍCH", "ĐỀN MIẾU", "TỪ MIẾU", "LĂNG MỘ", "ĐỒN LŨY", "CẦU ĐƯỜNG", "CHỢ QUÁN",
]

DOMAIN_TOKENS = set(" ".join(DOMAIN_HEADINGS).lower().split()) | COMMON_VI_WORDS

LIBRARY_NOISE_RE = re.compile(
    r"\b(library|libraries|university|barcode|digitized|google|riverside|michigan|wisconsin|madison|"
    r"east\s+west|center\s+library|memorial|archive|scan|call\s*number|state\s+street|copyright|"
    r"vol\.?|volume|DS\s*\d|G27|EAST WEST CENTER|THE UNIVERSITY)\b",
    re.I,
)
BARCODE_CALLNO_RE = re.compile(r"^(?:[A-Z]{0,3}\s*)?(?:DS|G|B|HV|VIET)?\s*[A-Z0-9.\-/ ]{2,18}$", re.I)
TITLE_FRONTMATTER_RE = re.compile(
    r"\b(văn\s*-?\s*hóa|tùng\s*-?\s*thư|dịch\s*-?\s*giả|soạn\s*-?\s*giả|xuất\s*-?\s*bản|"
    r"bộ\s+quốc\s*-?\s*gia|bộ\s+văn\s*-?\s*hóa|nhà\s+xuất\s+bản|thuận\s+hóa|tập\s+số|"
    r"tái\s+bản|người\s+dịch|người\s+hiệu\s+đính)\b",
    re.I,
)

BACKMATTER_NOISE_RE = re.compile(
    r"\b(Những\s+tập\s+VĂN\s*HÓA|Có\s+bán\s+khắp|Tổng\s*-?\s*phát\s*-?\s*hành|"
    r"NHA\s+VĂN\s*[-–]?\s*HÓA\s*\(\s*266|Đường\s+Công\s+Lý|In\s+50\s+cuốn|In\s+1000\s+cuốn|"
    r"Số\s+đăng\s+kí\s+KHXB|Quyết\s+định\s+xuất\s+bản|Xưởng\s+in\s+Ban|MỤC\s+LỤC|MUC\s+LUC)\b",
    re.I,
)

MOJIBAKE_RE = re.compile(r"[ÑñÐðÖ¿§€]")

NOISY_TESSERACT_TOKEN_RE = re.compile(
    r"\b(NÑam|NÑinh|giấp|s15|7oa|l3ình|trừ-tjch|khéng|lèo|"
    r"NLIAT|TERIOCNG|ALAIN|TA\s+T|xuat\s+bản|phat\s+xu|ngu[eé]n|huy[eé]n\s+nav)\b",
    re.I,
)

CONTENT_START_RE = re.compile(
    r"\b(lời\s+nói\s+đầu|bài\s+tự|phàm\s+lệ|quyền\s+[ivxlcdm0-9]+|tỉnh\s+[A-ZÀ-ỸĐ]|"
    r"dựng\s+đặt|diên\s+cách|diễn\s+cách|phần\s+dã|hình\s+thế|phong\s+tục|đông\s+tây\s+cách\s+nhau)\b",
    re.I,
)

CLEAR_JUNK_RE = re.compile(
    r"(OPOC|erererore|Fel\s+Fat|Seer\s+tit|mADS|OKOK|OROR|RORO|Peete|Sarine|"
    r"^[\s:;,.`´‘’\"\\/\-–—_~|°*+={}\[\]()<>]{1,14}$|^\s*5\s*[-–—]\s*$)",
    re.I,
)
REPEATED_FRAGMENT_RE = re.compile(r"([A-Za-z]{2,5})\1{2,}")

print("imports ok")


imports ok


In [4]:

# ============================================================
# 3. Download / discover PDFs
# ============================================================

def download_drive_url(url: str, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    if not url or "PASTE" in url:
        return
    print("Downloading from Drive:", url)
    try:
        if "/folders/" in url:
            gdown.download_folder(url, output=str(out_dir), quiet=False, use_cookies=False, remaining_ok=True)
        else:
            gdown.download(url, output=str(out_dir), quiet=False, fuzzy=True)
    except TypeError:
        # Older gdown fallback.
        if "/folders/" in url:
            subprocess.run([sys.executable, "-m", "gdown", "--folder", url, "-O", str(out_dir)], check=False)
        else:
            subprocess.run([sys.executable, "-m", "gdown", url, "-O", str(out_dir)], check=False)
    except Exception as e:
        print("Drive download failed:", repr(e))

RAW_DIR.mkdir(parents=True, exist_ok=True)
if DRIVE_URLS:
    existing = list(RAW_DIR.rglob("*.pdf"))
    if not existing:
        for url in DRIVE_URLS:
            download_drive_url(url, RAW_DIR)

pdf_candidates = []
pdf_candidates += list(RAW_DIR.rglob("*.pdf"))
if ALLOW_KAGGLE_INPUT_FALLBACK:
    for d in LOCAL_INPUT_DIRS:
        if d.exists():
            pdf_candidates += list(d.rglob("*.pdf"))

seen = set()
pdf_paths = []
for p in sorted(pdf_candidates):
    try:
        key = (p.name.lower(), p.stat().st_size)
        if key in seen:
            continue
        seen.add(key)
        pdf_paths.append(p)
    except Exception:
        pass

if FAST_TEST_MODE:
    pdf_paths = pdf_paths[:FAST_TEST_MAX_PDFS]

if not pdf_paths:
    raise FileNotFoundError("No PDFs found. Check DRIVE_URLS, Internet=On, or /kaggle/input.")

print("PDF count:", len(pdf_paths))
for p in pdf_paths[:60]:
    print("-", p)
if len(pdf_paths) > 60:
    print("...", len(pdf_paths) - 60, "more")


Retrieving folder contents


Processing file 1wukxPsU1Ty_vWeGSk6CdvEZwLAcjdnqh 01.pdf
Processing file 1NLyeEqQCMW5-fnz797Oxx7R2f6X3o170 05.pdf
Processing file 1unk05e2iCFeaxSVupNam5DcDD5brtF2E 07_08.pdf
Processing file 16FEI4ljzrewz5bvt8tMAI8xQU4Ecut9h 09.pdf
Processing file 1C-mYRYagVEcEOyKCS93I0ZHaBpUw8_Wr 10_11.pdf
Processing file 1ROEJnPaAspyXW3k81b4zPjN6TBazDrt- 12.pdf
Processing file 126X-S8gfQHznvYhifiITln-xxy_h_r61 13.pdf
Processing file 1OK-e2fx66HxCDOF1ZoBpdG0qhy12-8qL 14_15.pdf
Processing file 1tFvtR94BGd1eIGI6jQOnR-z_nPqYbDFZ 16_17.pdf
Processing file 12KF-95e9GN1edzQxpTl3OfwZcntZgkV_ q2_3_4.pdf
Processing file 1rQB3SI4qvl7iJPnUq4Bo9en4db-7vaRv q6.pdf


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1wukxPsU1Ty_vWeGSk6CdvEZwLAcjdnqh
To: /kaggle/working/dntc_auto/raw_drive/01.pdf
100%|██████████| 6.92M/6.92M [00:00<00:00, 27.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1NLyeEqQCMW5-fnz797Oxx7R2f6X3o170
To: /kaggle/working/dntc_auto/raw_drive/05.pdf
100%|██████████| 23.7M/23.7M [00:00<00:00, 50.3MB/s]
Downloading...
From: https://drive.google.com/uc?id=1unk05e2iCFeaxSVupNam5DcDD5brtF2E
To: /kaggle/working/dntc_auto/raw_drive/07_08.pdf
100%|██████████| 9.98M/9.98M [00:00<00:00, 155MB/s]
Downloading...
From: https://drive.google.com/uc?id=16FEI4ljzrewz5bvt8tMAI8xQU4Ecut9h
To: /kaggle/working/dntc_auto/raw_drive/09.pdf
100%|██████████| 8.60M/8.60M [00:00<00:00, 38.1MB/s]
Downloading...
From: https://drive.google.com/uc?id=1C-mYRYagVEcEOyKCS93I0ZHaBpUw8_Wr
To: /kaggle/working/dntc_auto/raw_drive/10_11.pdf
100%|████████

PDF count: 11
- /kaggle/working/dntc_auto/raw_drive/01.pdf
- /kaggle/working/dntc_auto/raw_drive/05.pdf
- /kaggle/working/dntc_auto/raw_drive/07_08.pdf
- /kaggle/working/dntc_auto/raw_drive/09.pdf
- /kaggle/working/dntc_auto/raw_drive/10_11.pdf
- /kaggle/working/dntc_auto/raw_drive/12.pdf
- /kaggle/working/dntc_auto/raw_drive/13.pdf
- /kaggle/working/dntc_auto/raw_drive/14_15.pdf
- /kaggle/working/dntc_auto/raw_drive/16_17.pdf
- /kaggle/working/dntc_auto/raw_drive/q2_3_4.pdf
- /kaggle/working/dntc_auto/raw_drive/q6.pdf



Download completed


In [5]:

# ============================================================
# 4. Text normalization, metrics, quality score
# ============================================================

def norm_text(s) -> str:
    if s is None or (isinstance(s, float) and math.isnan(s)):
        return ""
    s = str(s)
    s = unicodedata.normalize("NFC", s)
    replacements = {
        "\u00a0": " ", "\ufeff": " ", "￾": " ", "ﬁ": "fi", "ﬂ": "fl",
        "`": "'", "´": "'", "“": "\"", "”": "\"", "‘": "'", "’": "'",
    }
    for a, b in replacements.items():
        s = s.replace(a, b)
    s = CONTROL_RE.sub(" ", s)
    s = s.replace("\r", " ").replace("\n", " ")
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def normalize_for_match(s: str) -> str:
    s = norm_text(s).lower()
    s = re.sub(r"[\-–—_]+", " ", s)
    s = re.sub(r"\s+", " ", s)
    return s.strip()


def strip_accents(s: str) -> str:
    s = unicodedata.normalize("NFD", s)
    return "".join(ch for ch in s if unicodedata.category(ch) != "Mn").replace("đ", "d").replace("Đ", "D")


def count_repeated_fragment_ratio(s: str) -> float:
    s0 = re.sub(r"\s+", "", norm_text(s))
    if len(s0) < 8:
        return 0.0
    repeated_chars = sum(len(m.group(0)) for m in REPEATED_FRAGMENT_RE.finditer(s0))
    # Also catch long runs like eeeee or -----.
    repeated_chars += sum(len(m.group(0)) for m in re.finditer(r"(.)\1{4,}", s0))
    return min(1.0, repeated_chars / max(1, len(s0)))


def punctuation_balance(s: str) -> float:
    s = norm_text(s)
    pairs = [("(", ")"), ("[", "]"), ("{", "}"), ("«", "»"), ('"', '"')]
    imbalance = 0
    for a, b in pairs:
        if a == b:
            imbalance += abs(s.count(a) % 2)
        else:
            imbalance += abs(s.count(a) - s.count(b))
    punct_density = len(re.findall(r"[.,;:!?/\\|]{2,}", s))
    return min(1.0, (imbalance + punct_density) / max(1, len(s) / 40.0))


def library_noise_score(s: str) -> float:
    s0 = norm_text(s)
    if not s0:
        return 0.0
    hits = len(LIBRARY_NOISE_RE.findall(s0))
    callno = 1 if BARCODE_CALLNO_RE.fullmatch(s0) and not CONTENT_START_RE.search(s0) else 0
    digit_chunks = len(re.findall(r"\d{4,}", s0))
    score = 0.17 * hits + 0.12 * callno + 0.025 * digit_chunks
    return min(1.0, score)


def latin_noise_ratio(s: str) -> float:
    s0 = norm_text(s)
    words = WORDS_RE.findall(s0)
    if not words:
        return 0.0
    bad = 0
    for w in words:
        wl = w.lower()
        if wl in COMMON_VI_WORDS or wl in COMMON_UNACCENTED_VI_WORDS:
            continue
        if len(w) >= 4:
            vowel_count = sum(1 for ch in w if ch in VIET_VOWELS)
            if vowel_count == 0:
                bad += 1
            elif ord(max(w)) < 128 and len(w) >= 7 and vowel_count / len(w) < 0.22:
                bad += 1
    return bad / max(1, len(words))


def text_metrics(text: str, lines=None, ocr_conf_values=None) -> dict:
    s = norm_text(text)
    n = max(1, len(s))
    letters = LETTERS_RE.findall(s)
    words = [w.lower() for w in WORDS_RE.findall(s)]
    word_count = len(words)
    accent_count = sum(1 for ch in s if ch in VIET_CHARS)
    cjk_count = len(CJK_RE.findall(s))
    weird_count = len(WEIRD_RE.findall(s)) + 3 * len(NON_VIET_SCRIPT_RE.findall(s))
    controls = len(CONTROL_RE.findall(str(text or "")))
    digits = len(DIGIT_RE.findall(s))
    dictionary_hits = sum(1 for w in words if w in COMMON_VI_WORDS)
    unaccented_hits = sum(1 for w in words if w in COMMON_UNACCENTED_VI_WORDS)
    domain_hits = sum(1 for h in DOMAIN_HEADINGS if h.lower() in s.lower())
    vietnamese_ratio = min(1.0, (dictionary_hits + 0.45 * accent_count + 2.0 * domain_hits) / max(1, word_count))
    dictionary_hit_ratio = dictionary_hits / max(1, word_count)
    accent_ratio = accent_count / max(1, len(letters))
    weird_char_ratio = (weird_count + controls) / n
    digit_ratio = digits / n
    rep_ratio = count_repeated_fragment_ratio(s)
    lib_score = library_noise_score(s)
    lat_noise = latin_noise_ratio(s)
    avg_line_length = 0.0
    if lines:
        nonempty = [norm_text(x) for x in lines if norm_text(x)]
        avg_line_length = float(np.mean([len(x) for x in nonempty])) if nonempty else 0.0
    else:
        avg_line_length = len(s)
    punct_bal = punctuation_balance(s)
    avg_ocr_conf = None
    if ocr_conf_values:
        vals = [float(x) for x in ocr_conf_values if x is not None and not pd.isna(x) and float(x) >= 0]
        avg_ocr_conf = float(np.mean(vals)) if vals else None

    # Quality score 0-100-ish. Designed to be conservative.
    score = 50.0
    score += 24.0 * dictionary_hit_ratio
    score += 18.0 * vietnamese_ratio
    score += min(8.0, avg_line_length / 12.0)
    if word_count >= 20:
        score += 4.0
    if accent_ratio >= 0.04:
        score += 4.0
    score -= 95.0 * weird_char_ratio
    score -= 28.0 * lat_noise
    score -= 30.0 * rep_ratio
    score -= 24.0 * lib_score
    score -= 12.0 * punct_bal
    score -= 16.0 * digit_ratio if digit_ratio > 0.22 else 0.0
    if word_count < 4 and not domain_hits:
        score -= 28.0
    if len(letters) < 15 and not domain_hits:
        score -= 18.0
    if word_count >= 12 and accent_ratio < 0.018 and unaccented_hits < 2 and dictionary_hits < 2:
        score -= 14.0
    if avg_ocr_conf is not None:
        if avg_ocr_conf < 25:
            score -= 16.0
        elif avg_ocr_conf < 38:
            score -= 8.0
        elif avg_ocr_conf > 55:
            score += 3.0
    if CLEAR_JUNK_RE.search(s):
        score -= 32.0
    if CONTENT_START_RE.search(s):
        score += 5.0
    return {
        "char_count": len(s), "letter_count": len(letters), "word_count": word_count,
        "accent_count": accent_count, "accent_ratio": accent_ratio, "cjk_count": cjk_count,
        "weird_char_count": weird_count, "control_count": controls, "weird_char_ratio": weird_char_ratio,
        "vietnamese_ratio": vietnamese_ratio, "dictionary_hit_ratio": dictionary_hit_ratio,
        "unaccented_vi_hits": unaccented_hits, "domain_heading_hits": domain_hits,
        "latin_noise_ratio": lat_noise, "avg_line_length": avg_line_length,
        "punctuation_balance": punct_bal, "digit_ratio": digit_ratio,
        "repeated_char_ngram_ratio": rep_ratio, "library_noise_score": lib_score,
        "avg_ocr_conf": avg_ocr_conf, "quality_score": round(score, 3),
    }


def is_heading_text(s: str) -> bool:
    s0 = norm_text(s)
    if not s0 or len(s0) > 110:
        return False
    su = s0.upper()
    if any(h in su for h in DOMAIN_HEADINGS):
        return True
    if re.fullmatch(r"(?:QUYỂN|QUYEN|TẬP|TAP|TỈNH|TINH)\s+[A-ZÀ-ỸĐ0-9IVXLCDM .\-]+", su):
        return True
    letters = LETTERS_RE.findall(s0)
    if len(letters) >= 4:
        upperish = sum(1 for ch in letters if ch.upper() == ch) / len(letters)
        if upperish >= 0.82 and len(s0) <= 80 and not LIBRARY_NOISE_RE.search(s0):
            return True
    return False


def is_title_or_frontmatter_text(s: str, page_number: int) -> bool:
    s0 = norm_text(s)
    if not s0:
        return False
    if page_number <= AUTO_CONTENT_SCAN_PAGES and TITLE_FRONTMATTER_RE.search(s0):
        # A long actual preface page can mention publisher; do not classify as title if content terms dominate.
        words = WORDS_RE.findall(s0)
        if len(words) < 70 or not CONTENT_START_RE.search(s0):
            return True
    title_only_tokens = ["ĐẠI NAM", "NHẤT THỐNG", "DỊCH GIẢ", "XUẤT BẢN", "NHÀ XUẤT BẢN", "TẬP SỐ"]
    hit = sum(1 for t in title_only_tokens if t.lower() in s0.lower())
    if page_number <= AUTO_CONTENT_SCAN_PAGES and hit >= 2 and len(WORDS_RE.findall(s0)) < 80:
        return True
    return False


In [6]:

# ============================================================
# 5. Rendering, visual page metrics, OCR, PDF text extraction
# ============================================================

def tesseract_available() -> bool:
    return ENABLE_TESSERACT and pytesseract is not None and shutil.which("tesseract") is not None


def render_page_to_pil(page, dpi=90, clip=None) -> Image.Image:
    zoom = dpi / 72.0
    pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), clip=clip, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


def visual_page_metrics(page) -> dict:
    try:
        img = render_page_to_pil(page, dpi=45)
        arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
        gray = arr.mean(axis=2)
        whiteness = float(np.mean(gray > 0.94))
        darkness = float(np.mean(gray < 0.12))
        ink_ratio = float(np.mean(gray < 0.88))
        color_std = float(np.mean(np.std(arr, axis=(0, 1))))
        mx = arr.max(axis=2)
        mn = arr.min(axis=2)
        saturation = float(np.mean((mx - mn) / np.maximum(mx, 1e-6)))
        gy = np.abs(np.diff(gray, axis=0)).mean()
        gx = np.abs(np.diff(gray, axis=1)).mean()
        edge_density = float(gx + gy)
        h, w = gray.shape
        border = max(2, int(min(h, w) * 0.035))
        border_pixels = np.concatenate([
            gray[:border, :].ravel(), gray[-border:, :].ravel(), gray[:, :border].ravel(), gray[:, -border:].ravel()
        ])
        border_ink_ratio = float(np.mean(border_pixels < 0.80))
        return {
            "white_ratio": whiteness, "dark_ratio": darkness, "ink_ratio": ink_ratio,
            "color_std": color_std, "saturation": saturation, "edge_density": edge_density,
            "border_ink_ratio": border_ink_ratio,
            "visual_blank_score": float(whiteness > 0.985 and ink_ratio < 0.018),
            "visual_pattern_score": min(1.0, max(0.0, saturation * 1.8 + color_std * 1.2 + edge_density * 4.0 - whiteness * 0.75)),
        }
    except Exception as e:
        return {"visual_error": type(e).__name__, "white_ratio": None, "dark_ratio": None, "ink_ratio": None,
                "color_std": None, "saturation": None, "edge_density": None, "border_ink_ratio": None,
                "visual_blank_score": 0.0, "visual_pattern_score": 0.0}


def pil_preprocess_for_ocr(img: Image.Image, mode="page") -> Image.Image:
    img = img.convert("RGB")
    gray = ImageOps.grayscale(img)
    gray = ImageOps.autocontrast(gray)
    gray = ImageEnhance.Contrast(gray).enhance(1.35 if mode == "page" else 1.65)
    gray = ImageEnhance.Sharpness(gray).enhance(1.20 if mode == "page" else 1.55)
    if mode == "line":
        gray = gray.filter(ImageFilter.MedianFilter(size=3))
    return gray


def tesseract_data_from_image(img: Image.Image, psm=6):
    if not tesseract_available():
        return pd.DataFrame()
    try:
        config = f"--oem 1 --psm {psm}"
        df = pytesseract.image_to_data(img, lang=OCR_LANG, config=config, output_type=Output.DATAFRAME)
        if df is None:
            return pd.DataFrame()
        return df
    except Exception as e:
        return pd.DataFrame()


def extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=PAGE_OCR_DPI) -> list:
    if not tesseract_available():
        return []
    img = render_page_to_pil(page, dpi=dpi)
    zoom = dpi / 72.0
    img = pil_preprocess_for_ocr(img, mode="page")
    df = tesseract_data_from_image(img, psm=6)
    if df.empty:
        return []
    df = df.dropna(subset=["text"]).copy()
    if df.empty:
        return []
    df["text"] = df["text"].map(norm_text)
    df = df[df["text"].str.len() > 0]
    if df.empty:
        return []
    if "conf" in df.columns:
        df["conf_num"] = pd.to_numeric(df["conf"], errors="coerce").fillna(-1)
    else:
        df["conf_num"] = -1
    group_cols = [c for c in ["block_num", "par_num", "line_num"] if c in df.columns]
    if not group_cols:
        group_cols = ["level"] if "level" in df.columns else []
    if not group_cols:
        return []
    lines = []
    for _, g in df.groupby(group_cols, sort=True):
        words = [norm_text(x) for x in g["text"].tolist() if norm_text(x)]
        text = norm_text(" ".join(words))
        if not text:
            continue
        conf_vals = [float(x) for x in g["conf_num"].tolist() if float(x) >= 0]
        conf = float(np.mean(conf_vals)) if conf_vals else -1.0
        left = float(g["left"].min()) / zoom
        top = float(g["top"].min()) / zoom
        right = float((g["left"] + g["width"]).max()) / zoom
        bottom = float((g["top"] + g["height"]).max()) / zoom
        lines.append({
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_idx + 1,
            "source": "tesseract_page", "bbox": [left, top, right, bottom], "raw_text": text,
            "ocr_conf": round(conf, 3), "block_id": None, "line_id": None,
        })
    lines.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return lines


def extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path) -> list:
    out = []
    try:
        data = page.get_text("dict")
    except Exception:
        return out
    for b_i, block in enumerate(data.get("blocks", [])):
        if block.get("type") != 0:
            continue
        for l_i, line in enumerate(block.get("lines", [])):
            spans = line.get("spans", [])
            text = norm_text(" ".join([sp.get("text", "") for sp in spans]))
            if not text:
                continue
            bbox = line.get("bbox", block.get("bbox", None))
            if not bbox:
                continue
            font_sizes = [float(sp.get("size", 0) or 0) for sp in spans]
            out.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_idx + 1,
                "source": "pdf_text_layer", "bbox": [float(x) for x in bbox], "raw_text": text,
                "ocr_conf": None, "block_id": b_i, "line_id": l_i,
                "font_size_avg": round(float(np.mean(font_sizes)) if font_sizes else 0.0, 3),
            })
    out.sort(key=lambda r: (r["bbox"][1], r["bbox"][0]))
    return out


def lines_to_text(lines) -> str:
    return "\n".join(norm_text(r.get("raw_text") or r.get("text")) for r in lines if norm_text(r.get("raw_text") or r.get("text")))


def line_crop_ocr(page, bbox) -> tuple:
    if not tesseract_available():
        return "", -1.0
    r = fitz.Rect(bbox)
    r.x0 = max(page.rect.x0, r.x0 - LINE_REOCR_PAD_PT)
    r.y0 = max(page.rect.y0, r.y0 - LINE_REOCR_PAD_PT)
    r.x1 = min(page.rect.x1, r.x1 + LINE_REOCR_PAD_PT)
    r.y1 = min(page.rect.y1, r.y1 + LINE_REOCR_PAD_PT)
    pix = page.get_pixmap(matrix=fitz.Matrix(LINE_REOCR_ZOOM, LINE_REOCR_ZOOM), clip=r, alpha=False)
    img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
    variants = [pil_preprocess_for_ocr(img, mode="line"), img]
    best_text, best_conf, best_score = "", -1.0, -999.0
    for im in variants:
        df = tesseract_data_from_image(im, psm=7)
        if df.empty:
            continue
        df = df.dropna(subset=["text"]).copy()
        if df.empty:
            continue
        txt = norm_text(" ".join(df["text"].map(norm_text).tolist()))
        conf = -1.0
        if "conf" in df.columns:
            vals = pd.to_numeric(df["conf"], errors="coerce")
            vals = vals[vals >= 0]
            if len(vals):
                conf = float(vals.mean())
        score = text_metrics(txt, ocr_conf_values=[conf]).get("quality_score", -999.0)
        if score > best_score:
            best_text, best_conf, best_score = txt, conf, score
    return best_text, best_conf


In [7]:

# ============================================================
# 6. Conservative Vietnamese/domain correction with log
# ============================================================
CORRECTION_RULES = [
    # Headings and title variants.
    ("heading_dai_nam", r"\bDAI\s*-?\s*NAM\b|\bĐAI\s*-?\s*NAM\b", "ĐẠI NAM"),
    ("heading_nhat_thong_chi", r"\bNH[ẤA]T\s*-?\s*TH[OỐ]NG\s*-?\s*CH[ÍI]\b|\bNHAT\s+THONG\s+CHI\b", "NHẤT THỐNG CHÍ"),
    ("heading_quyen", r"\bQUYEN\b", "QUYỂN"),
    ("heading_tinh", r"\bTINH\b(?=\s+[A-ZÀ-ỸĐ])", "TỈNH"),
    ("heading_phan_da", r"\bPHAN\s+DA\b|\bPHẦN\s+DA\b", "PHẦN DÃ"),
    ("heading_dung_dat", r"\bD[UƯ]NG\s+[DPĐ]AT\s+VA\s+DI[ÊE]N\s+C[ÁA]CH\b|\bDUNG\s+DAT\s+VA\s+DIEN\s+CACH\b", "DỰNG ĐẶT VÀ DIÊN CÁCH"),
    ("heading_hinh_the", r"\bHINH\s+THE\b", "HÌNH THẾ"),
    ("heading_phong_tuc", r"\bPHONG\s+TUC\b", "PHONG TỤC"),
    ("heading_thue_ruong", r"\bTHUE\s+RUONG\b", "THUẾ RUỘNG"),
    ("heading_nui_song", r"\bNUI\s+SONG\b", "NÚI SÔNG"),

    # Specific OCR/mojibake artifacts reported for DNTC.
    ("mojibake_chiem", r"\bchi€m\b", "chiếm"),
    ("mojibake_bien", r"\bbi€n\b", "biển"),
    ("mojibake_kiem", r"\bki€m\b", "kiêm"),
    ("mojibake_mieu", r"\bmi€u\b", "miếu"),
    ("n_tilde_nam", r"\bñăm\b", "năm"),
    ("the_ky", r"\bth[eéế]\s+k[yỷ]\b|\bth[eéế]\s+kỷ\b", "thế kỷ"),
    ("dau_the_ky", r"\bĐầu\s+th[eéế]\s+k[yỷ]\b", "Đầu thế kỷ"),
    ("doi_tu_duc", r"\bdoi\s+#?ự\s+Đức\b|\bđời\s+#?ự\s+Đức\b", "đời Tự Đức"),
    ("doi_hien_tong", r"\bdoi\s+H[ií]én\s+Tông\b", "đời Hiến Tông"),
    ("nha_tuy", r"\bNha\s+Tuy\b", "Nhà Tùy"),
    ("nha_duong", r"\bNha\s+Đường\b", "Nhà Đường"),
    ("nuoc_ta", r"\bNước\s+tả\b", "Nước ta"),
    ("cao_men", r"\bCao\s+M[eé]n\b", "Cao Mên"),
    ("huyen_hien", r"\bhuyénhién\b", "huyện hiện"),
    ("cuop", r"\bcudp\b", "cướp"),


    # Additional systematic artifacts from the second full run.
    ("mojibake_nguyen_upper", r"\bÑg", "Ng"),
    ("mojibake_n_upper", r"\bÑ(?=[A-ZÀ-ỸĐa-zà-ỹđ])", "N"),
    ("mojibake_n_lower", r"ñ", "n"),
    ("mojibake_d_upper", r"Ð", "Đ"),
    ("mojibake_d_lower", r"ð", "đ"),
    ("mojibake_o_place", r"\bÖ['’]?\s+(?=phía|thôn|xã|huyện|trên|cực|địa)", "Ở "),
    ("ocr_zero_place", r"\b0['°]\s+(?=phía|thôn|xã|huyện|trên|cực|địa)", "Ở "),
    ("dam_dim_after_number", r"(\d+)\s+d[ıi]m\b", r"\1 dặm"),
    ("dam_mojibake_after_number", r"(\d+)\s+d[§s]\s*m\b", r"\1 dặm"),
    ("tir_tinh_li", r"\bTir\s+t[ií]nh\s*[- ]\s*l[ií]\b", "Từ tỉnh-lỵ"),
    ("tinh_li_hyphen", r"\btinh\s*[- ]\s*li\b", "tỉnh-lỵ"),
    ("le_thanh_tong", r"\bLe\s+Thdnh\s+T[eé]ng\b", "Lê Thánh Tông"),
    ("thi_si", r"\bThi\s+Si\b", "Thi Sĩ"),
    ("hong_duc", r"\bHong\s+Đức\b", "Hồng Đức"),
    ("gia_cat", r"\bCia\s*[- ]\s*cat\b", "Gia Cát"),
    ("thing_chi", r"\bthing\s+chí\b", "thống chí"),
    ("word_chonay", r"\bchỗnày\b", "chỗ này"),
    ("word_chonao", r"\bchỗnào\b", "chỗ nào"),
    ("word_canui", r"\bcảnúi\b", "cả núi"),
    ("word_nhantai", r"\bnhântài\b", "nhân tài"),
    ("word_nienhieu", r"\bniênhiệu\b", "niên hiệu"),
    ("word_vephia", r"\bvềphía\b", "về phía"),
    ("word_tuphia", r"\btừphía\b", "từ phía"),

    # Contextual administrative words. Conservative: only in common administrative contexts.
    ("tinh_ly", r"\btinh\s+l[yỵi]\b", "tỉnh lỵ"),
    ("tinh_thanh", r"\btinh\s+thành\b", "tỉnh thành"),
    ("tinh_place", r"\btinh\s+(Quảng|Thanh|Nghệ|Bình|Phú|Khánh|Hà|An|Gia|Định|Vĩnh|Biên|Quy|Hải|Nam|Bắc)\b", r"tỉnh \1"),
    ("huyen_context", r"\bhuyen\s+(?=[A-ZÀ-ỸĐ])", "huyện "),
    ("phu_context", r"\bphu\s+(?=[A-ZÀ-ỸĐ])", "phủ "),
    ("chau_context", r"\bchau\s+(?=[A-ZÀ-ỸĐ])", "châu "),

    # Common names.
    ("minh_menh", r"\bMinh\s+M[ée]nh\b|\bMinh\s+Mộệnh\b", "Minh Mệnh"),
    ("thieu_tri", r"\bThiệu\s+Tri\b", "Thiệu Trị"),
    ("tu_duc", r"\bTự\s+Dức\b|\bDự\s+Đức\b", "Tự Đức"),
    ("gia_du", r"\bGia\s+Du\s+Hoàng\b", "Gia Dụ Hoàng"),
    ("ha_tien", r"\bHa\s+Tien\b", "Hà Tiên"),
    ("quang_binh", r"\bQuang\s+Binh\b", "Quảng Bình"),
    ("binh_dinh", r"\bBinh\s+Dinh\b", "Bình Định"),
    ("quang_yen", r"\bQuang\s+Yen\b", "Quảng Yên"),
]


def apply_corrections(text: str, meta: dict, correction_log: list) -> str:
    original = norm_text(text)
    s = original
    # First normalize punctuation and known glyphs.
    pre = s
    s = s.replace("￾", " ").replace("\u0001", " ")
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,.;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(\d)\s+([,.])\s+(\d)", r"\1\2\3", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s != pre:
        correction_log.append({**meta, "level": "normalize", "rule_id": "unicode_space_punct", "before": pre, "after": s})
    for rule_id, pat, repl in CORRECTION_RULES:
        before = s
        s, n = re.subn(pat, repl, s, flags=re.IGNORECASE)
        if n and s != before:
            correction_log.append({**meta, "level": "domain_rule", "rule_id": rule_id, "before": before, "after": s})
    s = re.sub(r"\s+", " ", s).strip()
    return s


def candidate_correction_only(text: str) -> str:
    tmp_log = []
    return apply_corrections(text, {}, tmp_log)


def choose_better_line_text(base_text: str, ocr_text: str, ocr_conf: float, meta: dict, correction_log: list) -> tuple:
    base_fixed = apply_corrections(base_text, meta, correction_log)
    ocr_fixed = candidate_correction_only(ocr_text)
    if not ocr_fixed:
        return base_fixed, "rules_only", False, ""
    base_m = text_metrics(base_fixed)
    ocr_m = text_metrics(ocr_fixed, ocr_conf_values=[ocr_conf])
    # Keep base if OCR loses too many digits or becomes much shorter.
    base_digits = re.findall(r"\d+(?:[.,]\d+)?", base_fixed)
    if base_digits:
        kept = sum(1 for d in base_digits if d in ocr_fixed)
        if kept / max(1, len(base_digits)) < 0.70:
            return base_fixed, "keep_base_ocr_lost_digits", False, ocr_fixed
    if len(ocr_fixed) < 0.55 * max(1, len(base_fixed)):
        return base_fixed, "keep_base_ocr_too_short", False, ocr_fixed
    if ocr_conf is not None and ocr_conf >= 0 and ocr_conf < 28 and ocr_m["quality_score"] < base_m["quality_score"] + 10:
        return base_fixed, "keep_base_low_ocr_conf", False, ocr_fixed
    if ocr_m["quality_score"] >= base_m["quality_score"] + 8 and ocr_m["weird_char_ratio"] <= base_m["weird_char_ratio"] + 0.02:
        # Log accepted line OCR as a correction event.
        correction_log.append({**meta, "level": "line_ocr", "rule_id": "accepted_better_line_ocr", "before": base_fixed, "after": ocr_fixed})
        return ocr_fixed, f"accept_line_ocr {base_m['quality_score']:.1f}->{ocr_m['quality_score']:.1f} conf={ocr_conf:.1f}", True, ocr_fixed
    return base_fixed, f"keep_base {base_m['quality_score']:.1f}->{ocr_m['quality_score']:.1f} conf={ocr_conf:.1f}", False, ocr_fixed


In [8]:

# ============================================================
# 7. Page classification and source selection
# ============================================================

def is_blank_page(text_m: dict, visual_m: dict) -> bool:
    if text_m["word_count"] <= 2 and (visual_m.get("visual_blank_score") == 1.0 or (visual_m.get("white_ratio") or 0) > 0.975):
        return True
    if text_m["letter_count"] < 5 and (visual_m.get("ink_ratio") or 1) < 0.025:
        return True
    return False


def is_patterned_page(text_m: dict, visual_m: dict, page_number: int) -> bool:
    if page_number > AUTO_CONTENT_SCAN_PAGES and text_m["word_count"] > 12:
        return False
    pattern_visual = (visual_m.get("visual_pattern_score") or 0) > 0.38 and (visual_m.get("white_ratio") or 1) < 0.82
    bad_text = text_m["vietnamese_ratio"] < 0.12 and text_m["word_count"] < 45
    return bool(pattern_visual and bad_text)



def is_backmatter_page(text: str, text_m: dict) -> bool:
    """Publisher ads / print info / TOC pages after main content.
    Keep real content containing domain headings; drop pages dominated by publishing metadata."""
    s = norm_text(text)
    if not s:
        return False
    if not BACKMATTER_NOISE_RE.search(s):
        return False
    # Mục lục / print info / sales ads are never body content.
    if re.search(r"\b(MỤC\s+LỤC|MUC\s+LUC|Số\s+đăng\s+kí\s+KHXB|Quyết\s+định\s+xuất\s+bản|In\s+(50|1000)\s+cuốn|Những\s+tập\s+VĂN\s*HÓA)\b", s, re.I):
        return True
    if text_m.get("word_count", 0) < 120 and text_m.get("domain_heading_hits", 0) == 0:
        return True
    return False


def is_library_page(text: str, text_m: dict) -> bool:
    if LIBRARY_NOISE_RE.search(norm_text(text)):
        return True
    if text_m["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE and text_m["vietnamese_ratio"] < 0.25:
        return True
    return False


def page_content_signal(text: str, text_m: dict) -> float:
    s = norm_text(text)
    score = 0.0
    score += min(35.0, text_m["word_count"] * 0.45)
    score += 25.0 * text_m["vietnamese_ratio"]
    score += 18.0 if CONTENT_START_RE.search(s) else 0.0
    score += 10.0 if any(h.lower() in s.lower() for h in DOMAIN_HEADINGS) else 0.0
    score -= 22.0 if TITLE_FRONTMATTER_RE.search(s) and text_m["word_count"] < 70 else 0.0
    score -= 35.0 * text_m["library_noise_score"]
    score -= 18.0 * text_m["repeated_char_ngram_ratio"]
    return score


def classify_page(page_number: int, text: str, text_m: dict, visual_m: dict, content_start_page: int | None) -> tuple:
    reasons = []
    cls = "content_candidate"
    if DROP_BLANK_PAGES and is_blank_page(text_m, visual_m):
        return "blank_page", ["blank_visual_or_no_text"]
    if DROP_PATTERNED_PAGES and is_patterned_page(text_m, visual_m, page_number):
        return "patterned_endpaper", ["patterned_visual_low_text"]
    if DROP_LIBRARY_PAGES and is_library_page(text, text_m):
        return "library_barcode_stamp_watermark", ["library_barcode_google_tokens"]
    if globals().get("DROP_BACKMATTER_PAGES", True) and is_backmatter_page(text, text_m):
        return "publisher_backmatter_page", ["publisher_backmatter_tokens"]
    if content_start_page is not None and DROP_FRONT_MATTER_BEFORE_CONTENT and page_number < content_start_page:
        return "front_matter_before_content", [f"before_auto_content_start_{content_start_page}"]
    if not KEEP_TITLE_PAGES and is_title_or_frontmatter_text(text, page_number):
        return "cover_title_front_matter", ["title_or_publisher_page"]
    if text_m["quality_score"] < 25 and text_m["word_count"] < 15:
        return "junk_ocr_page", ["low_text_quality_too_few_words"]
    return cls, reasons


def estimate_content_start(doc, work_id, pdf_path) -> int:
    if CONTENT_START_MODE != "auto":
        return 1
    max_scan = min(len(doc), AUTO_CONTENT_SCAN_PAGES)
    candidates = []
    for page_idx in range(max_scan):
        page = doc[page_idx]
        # Use text layer first. If text layer is nearly empty/bad, do a cheap OCR pass for content-start detection only.
        tl_lines = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
        tl_text = lines_to_text(tl_lines)
        tl_m = text_metrics(tl_text, lines=[r.get("raw_text", "") for r in tl_lines])
        detection_text = tl_text
        detection_m = tl_m
        if tl_m["letter_count"] < 20 or tl_m["weird_char_ratio"] > 0.10:
            try:
                ocr_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=FAST_FRONTMATTER_OCR_DPI)
                ocr_text = lines_to_text(ocr_lines)
                ocr_m = text_metrics(ocr_text, lines=[r.get("raw_text", "") for r in ocr_lines], ocr_conf_values=[r.get("ocr_conf") for r in ocr_lines])
                if ocr_m["quality_score"] > detection_m["quality_score"]:
                    detection_text, detection_m = ocr_text, ocr_m
            except Exception:
                pass
        visual_m = visual_page_metrics(page)
        preliminary_cls, _ = classify_page(page_idx + 1, detection_text, detection_m, visual_m, content_start_page=None)
        signal = page_content_signal(detection_text, detection_m)
        has_start = CONTENT_START_RE.search(norm_text(detection_text)) is not None
        title_front = is_title_or_frontmatter_text(detection_text, page_idx + 1)
        candidates.append({
            "page_number": page_idx + 1, "signal": signal, "has_start": has_start,
            "word_count": detection_m["word_count"], "line_count": len(detection_text.splitlines()),
            "quality_score": detection_m["quality_score"], "preliminary_cls": preliminary_cls,
            "title_front": title_front,
        })
    # Prefer the first page that has actual prose/content, not title-only.
    for c in candidates:
        if c["preliminary_cls"] in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark"}:
            continue
        if c["title_front"] and not KEEP_TITLE_PAGES:
            continue
        if c["has_start"] and c["word_count"] >= MIN_CONTENT_WORDS_ON_START_PAGE and c["quality_score"] >= 30:
            return int(c["page_number"])
    for c in candidates:
        if c["preliminary_cls"] == "content_candidate" and c["signal"] >= 35 and c["word_count"] >= MIN_CONTENT_WORDS_ON_START_PAGE:
            return int(c["page_number"])
    # Safe fallback: first page after obvious front matter with decent words.
    for c in candidates:
        if c["preliminary_cls"] == "content_candidate" and not c["title_front"] and c["word_count"] >= 25:
            return int(c["page_number"])
    return 1


def select_page_source(page, page_idx, work_id, pdf_path, page_class) -> tuple:
    # Always inspect text layer. OCR only if needed.
    tl_lines = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
    tl_text = lines_to_text(tl_lines)
    tl_m = text_metrics(tl_text, lines=[r.get("raw_text", "") for r in tl_lines])

    should_ocr = ENABLE_TESSERACT and tesseract_available() and (
        tl_m["quality_score"] < MIN_TEXT_LAYER_GOOD_QUALITY or
        tl_m["letter_count"] < 40 or
        tl_m["weird_char_ratio"] > 0.06 or
        tl_m["repeated_char_ngram_ratio"] > 0.08
    )
    ocr_lines, ocr_text, ocr_m = [], "", None
    if should_ocr and page_class not in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark", "cover_title_front_matter", "front_matter_before_content"}:
        ocr_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=PAGE_OCR_DPI)
        ocr_text = lines_to_text(ocr_lines)
        ocr_m = text_metrics(ocr_text, lines=[r.get("raw_text", "") for r in ocr_lines], ocr_conf_values=[r.get("ocr_conf") for r in ocr_lines])

    # Source decision.
    if ocr_m is None or not ocr_lines:
        selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
        reason = "text_layer_only_or_ocr_unavailable"
    else:
        # Do not let OCR replace a good text layer unless it is clearly better.
        if tl_m["quality_score"] >= MIN_TEXT_LAYER_GOOD_QUALITY and tl_m["quality_score"] >= ocr_m["quality_score"] - 7:
            selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
            reason = f"keep_text_layer_good tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        elif ocr_m["quality_score"] >= tl_m["quality_score"] + 7 and ocr_m["quality_score"] >= MIN_OCR_PAGE_QUALITY:
            selected, source, selected_m = ocr_lines, "tesseract_page", ocr_m
            reason = f"use_ocr_better tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        elif tl_m["quality_score"] >= 35 and tl_m["word_count"] >= 8:
            selected, source, selected_m = tl_lines, "pdf_text_layer", tl_m
            reason = f"fallback_text_layer_ocr_not_good_enough tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
        else:
            selected, source, selected_m = [], "drop_no_reliable_source", max([tl_m, ocr_m], key=lambda x: x["quality_score"])
            reason = f"drop_both_sources_low tl={tl_m['quality_score']:.1f} ocr={ocr_m['quality_score']:.1f}"
    return selected, source, selected_m, tl_m, (ocr_m or {}), reason


In [9]:

# ============================================================
# 8. Line filtering, paragraph reflow, sentence splitting
# ============================================================

def line_quality(text: str, ocr_conf=None) -> dict:
    return text_metrics(text, lines=[text], ocr_conf_values=[ocr_conf] if ocr_conf is not None else None)


def is_repeated_junk_line(s: str) -> bool:
    s0 = norm_text(s)
    if not s0:
        return True
    if CLEAR_JUNK_RE.search(s0):
        return True
    if REPEATED_FRAGMENT_RE.search(re.sub(r"\s+", "", s0)):
        return True
    words = WORDS_RE.findall(s0)
    if words and len(words) <= 5:
        no_vowels = sum(1 for w in words if len(w) >= 3 and not any(ch in VIET_VOWELS for ch in w))
        if no_vowels / max(1, len(words)) > 0.55:
            return True
    # Patterned endpapers often OCR as B, ER, 83, 3, ॐ repeated.
    if re.fullmatch(r"[\sBЕER83()*0-9ॐఓజిஆ६]+", s0, flags=re.I):
        return True
    return False



def has_vietnamese_diacritic(s: str) -> bool:
    return any(ch in VIET_CHARS for ch in norm_text(s))


def is_ascii_short_junk_line(s: str) -> bool:
    s0 = norm_text(s)
    if re.search(r"\b(A ee Ra|Ave sy|BREF|AG TA|OR tA|xf An|thon Ta-My|Fel Fat|Seer tit|mADS|OPOC)\b", s0, re.I):
        return True
    if any(ord(ch) >= 128 for ch in s0) or has_vietnamese_diacritic(s0):
        return False
    ws = WORDS_RE.findall(s0)
    if len(ws) >= 3 and len(s0) < 90:
        short_chunks = sum(1 for w in ws if len(w) <= 4)
        upper_chunks = sum(1 for w in ws if w.isupper())
        if short_chunks / max(1, len(ws)) > 0.82 or upper_chunks / max(1, len(ws)) > 0.50:
            return True
    return False


def is_line_hard_noise(s: str) -> bool:
    s0 = norm_text(s)
    if BACKMATTER_NOISE_RE.search(s0):
        return True
    if is_ascii_short_junk_line(s0):
        return True
    if NOISY_TESSERACT_TOKEN_RE.search(s0) and not has_vietnamese_diacritic(s0) and len(s0) < 80:
        return True
    return False


def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    meta = {
        "work_id": line.get("work_id"), "pdf_path": line.get("pdf_path"), "page_number": line.get("page_number"),
        "page_idx": line.get("page_idx"), "line_id": line.get("line_global_id"), "source": line.get("source"),
    }
    reasons = []
    if not raw:
        return None, False, ["empty"]
    if is_line_hard_noise(raw):
        return None, False, ["hard_noise_or_backmatter_line"]
    if LIBRARY_NOISE_RE.search(raw):
        return None, False, ["library_barcode_google_line"]
    if re.fullmatch(r"\d{1,4}", raw):
        return None, False, ["page_number_only"]
    if BARCODE_CALLNO_RE.fullmatch(raw) and len(WORDS_RE.findall(raw)) <= 3 and not is_heading_text(raw):
        return None, False, ["call_number_or_barcode_fragment"]
    if is_repeated_junk_line(raw):
        return None, False, ["repeated_or_known_ocr_junk"]

    fixed = apply_corrections(raw, meta, correction_log)
    # Optional line re-OCR only for suspicious PDF text layer lines. Do not replace OCR-page lines with another OCR.
    line_ocr_text = ""
    line_ocr_conf = None
    line_ocr_accepted = False
    if (doc is not None and reocr_state is not None and line.get("source") == "pdf_text_layer" and
        reocr_state.get("used", 0) < MAX_LINE_REOCR_PER_PDF):
        q0 = line_quality(fixed)
        suspicious_for_reocr = (
            q0["quality_score"] < MIN_KEEP_LINE_QUALITY + 8 or q0["weird_char_ratio"] > 0.035 or
            q0["repeated_char_ngram_ratio"] > 0.04 or q0["unaccented_vi_hits"] >= 2 and q0["accent_ratio"] < 0.015
        )
        if suspicious_for_reocr:
            try:
                page = doc[line["page_idx"]]
                line_ocr_text, line_ocr_conf = line_crop_ocr(page, line["bbox"])
                reocr_state["used"] = reocr_state.get("used", 0) + 1
                fixed2, reason, accepted, line_ocr_text2 = choose_better_line_text(fixed, line_ocr_text, line_ocr_conf, meta, correction_log)
                if accepted:
                    fixed = fixed2
                    line_ocr_accepted = True
                reasons.append(reason)
            except Exception as e:
                reasons.append(f"line_reocr_failed:{type(e).__name__}")

    q = line_quality(fixed, line.get("ocr_conf"))
    heading = is_heading_text(fixed)
    if not heading:
        if len(fixed) < 7 or q["letter_count"] < 3:
            return None, False, reasons + ["too_short_not_heading"]
        if q["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE:
            return None, False, reasons + ["library_noise_score"]
        if q["weird_char_ratio"] > MAX_WEIRD_CHAR_RATIO:
            return None, False, reasons + ["too_many_weird_chars"]
        if q["repeated_char_ngram_ratio"] > 0.12:
            return None, False, reasons + ["repeated_ngram_ratio"]
        if q["word_count"] >= 7 and q["vietnamese_ratio"] < MIN_VIET_RATIO_FOR_LONG_TEXT and q["accent_ratio"] < 0.025:
            return None, False, reasons + ["low_vietnamese_ratio"]
        if line.get("source") == "tesseract_page" and line.get("ocr_conf") is not None and float(line.get("ocr_conf")) >= 0:
            if float(line.get("ocr_conf")) < MIN_OCR_CONF_LINE and q["quality_score"] < MIN_KEEP_LINE_QUALITY + 8:
                return None, False, reasons + ["low_tesseract_conf"]
        if q["quality_score"] < MIN_KEEP_LINE_QUALITY:
            return None, False, reasons + [f"low_line_quality:{q['quality_score']:.1f}"]

    kept = dict(line)
    kept["text"] = fixed
    kept["line_type"] = "heading" if heading else "body"
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    kept["line_ocr_text"] = line_ocr_text
    kept["line_ocr_conf"] = line_ocr_conf
    kept["line_ocr_accepted"] = line_ocr_accepted
    kept["filter_reasons"] = ";".join(reasons) if reasons else "kept"
    return kept, True, reasons


def median_line_height(lines) -> float:
    vals = []
    for r in lines:
        try:
            b = r["bbox"]
            vals.append(max(1.0, float(b[3]) - float(b[1])))
        except Exception:
            pass
    return float(np.median(vals)) if vals else 12.0


def is_footnote_line(line: dict, page_rect) -> bool:
    try:
        y0 = line["bbox"][1]
        fs = line.get("font_size_avg") or 0
        text = line.get("text", "")
        return (y0 > page_rect.height * 0.80 and (re.match(r"^\(?\d+\)|^\*", text) or (fs and fs < 9)))
    except Exception:
        return False


def ends_strong_sentenceish(t: str) -> bool:
    t = norm_text(t)
    if not t:
        return False
    return bool(re.search(r'[.!?…;:)"»\\]]$', t))


def begins_continuation(t: str) -> bool:
    t = norm_text(t).lstrip(' "\'“”‘’([{<«»—–-:;,.')
    return bool(t and re.match(r'^[a-zàáảãạằắẳẵặầấẩẫậèéẻẽẹềếểễệìíỉĩịòóỏõọồốổỗộờớởỡợùúủũụừứửữựỳýỷỹỵđ]', t))


def should_new_paragraph(prev: dict, cur: dict, med_h: float, page_rect) -> bool:
    pt, ct = prev.get("text", ""), cur.get("text", "")
    if prev.get("line_type") == "heading" or cur.get("line_type") == "heading":
        return True
    if prev.get("is_footnote") != cur.get("is_footnote"):
        return True
    if re.match(r"^\(?\d+\)|^\d+[.)]", ct):
        return True
    try:
        gap = cur["bbox"][1] - prev["bbox"][3]
        indent_delta = cur["bbox"][0] - prev["bbox"][0]
        # The old threshold 0.95 split normal scanned lines into pseudo-sentences.
        # Only start a new paragraph on a large vertical gap, or a clear indent after a completed sentence.
        if gap > med_h * 1.85:
            return ends_strong_sentenceish(pt) or not begins_continuation(ct)
        if indent_delta > med_h * 2.0 and len(pt) > 35 and ends_strong_sentenceish(pt):
            return True
    except Exception:
        pass
    return False

def join_paragraph_lines(lines: list) -> str:
    parts = []
    for r in lines:
        t = norm_text(r.get("text", ""))
        if not t:
            continue
        if not parts:
            parts.append(t)
            continue
        prev = parts[-1]
        # Remove true line-break hyphen only for lowercase/letter split words.
        if re.search(r"[a-zà-ỹđ]-$", prev) and re.match(r"^[a-zà-ỹđ]", t):
            parts[-1] = prev[:-1] + t
        else:
            parts.append(t)
    text = " ".join(parts)
    text = re.sub(r"\s+", " ", text).strip()
    # Make old spaced punctuation less noisy.
    text = re.sub(r"\s+([,.;:!?])", r"\1", text)
    text = re.sub(r"([,.;:!?])(?=\S)", r"\1 ", text)
    text = re.sub(r"\(\s+", "(", text)
    text = re.sub(r"\s+\)", ")", text)
    return text


def reflow_lines_to_paragraphs(lines: list, page_rect) -> list:
    if not lines:
        return []
    lines = sorted(lines, key=lambda r: (r["bbox"][1], r["bbox"][0]))
    for r in lines:
        r["is_footnote"] = is_footnote_line(r, page_rect)
    med_h = median_line_height(lines)
    groups, cur = [], []
    for r in lines:
        if not cur:
            cur = [r]
        elif should_new_paragraph(cur[-1], r, med_h, page_rect):
            groups.append(cur)
            cur = [r]
        else:
            cur.append(r)
    if cur:
        groups.append(cur)
    out = []
    for i, g in enumerate(groups):
        text = join_paragraph_lines(g)
        if not text:
            continue
        b = [min(x["bbox"][0] for x in g), min(x["bbox"][1] for x in g), max(x["bbox"][2] for x in g), max(x["bbox"][3] for x in g)]
        if all(x.get("line_type") == "heading" for x in g) and len(text) <= 130:
            ptype = "heading"
        elif any(x.get("is_footnote") for x in g):
            ptype = "footnote"
        else:
            ptype = "body"
        q = text_metrics(text, lines=[x.get("text", "") for x in g])
        out.append({
            "paragraph_index_in_page": i, "paragraph_type": ptype, "text": text, "bbox": b,
            "line_count": len(g), "source": ";".join(sorted(set(x.get("source", "") for x in g))),
            "quality_score": q["quality_score"], "vietnamese_ratio": q["vietnamese_ratio"],
            "weird_char_ratio": q["weird_char_ratio"], "line_ids": ";".join(x.get("line_global_id", "") for x in g),
        })
    return out

ABBREV_PATTERNS = [
    "v.v.", "v.v..", "tr.C.N.", "T.P.", "P.", "S.", "HV.", "q.", "sđd.", "x.", "X.",
]


def protect_sentence_abbrevs(text: str) -> tuple:
    repl = {}
    protected = text
    # protect abbreviation dots
    for i, ab in enumerate(ABBREV_PATTERNS):
        key = f"§ABBR{i}§"
        protected = protected.replace(ab, key)
        repl[key] = ab
    # protect decimals and numbered references like A.69, HV.140.
    def repl_match(m):
        key = f"§DOT{len(repl)}§"
        repl[key] = m.group(0)
        return key
    protected = re.sub(r"\b[A-Z]{1,4}\.\d+\b", repl_match, protected)
    protected = re.sub(r"\b\d+\.\d+\b", repl_match, protected)
    return protected, repl


def unprotect_sentence_abbrevs(text: str, repl: dict) -> str:
    for k, v in repl.items():
        text = text.replace(k, v)
    return text


def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    text = norm_text(paragraph_text)
    if paragraph_type == "heading":
        return []
    if not text:
        return []
    protected, repl = protect_sentence_abbrevs(text)
    out = []
    buf = []
    depth = 0
    quote_open = False
    for i, ch in enumerate(protected):
        buf.append(ch)
        if ch in "([{" or ch == "«":
            depth += 1
        elif ch in ")]}" or ch == "»":
            depth = max(0, depth - 1)
        elif ch == '"':
            quote_open = not quote_open
        if ch in ".?!…;":
            tail = "".join(buf).strip()
            nxt = protected[i + 1:i + 8]
            next_char = protected[i + 1:i + 2]
            # Semicolon split only when sentence is already very long and next token looks like a new clause.
            if ch == ";" and len(tail) < 220:
                continue
            if depth > 0 and ch != ";":
                continue
            # Avoid splitting at numbered list/footnote markers or short fragments.
            if re.search(r"\(\s*\d+\s*\)$", tail):
                continue
            if next_char and not re.match(r"\s", next_char):
                continue
            # Find first non-space after punctuation.
            rest = protected[i + 1:]
            m = re.search(r"\S", rest)
            if m:
                nchar = rest[m.start()]
                if ch != ";" and not re.match(r"[A-ZÀ-ỸĐ0-9\(\"'«]", nchar):
                    continue
            sent = unprotect_sentence_abbrevs(tail, repl)
            sent = norm_text(sent)
            if sent:
                out.append(sent)
            buf = []
    remain = unprotect_sentence_abbrevs("".join(buf).strip(), repl)
    remain = norm_text(remain)
    if remain:
        # Sentence-only output must not leak line fragments. Merge non-terminal/lowercase remainders back.
        if out and (len(remain) < 60 or begins_continuation(remain) or not ends_strong_sentenceish(remain)):
            out[-1] = norm_text(out[-1] + " " + remain)
        elif not globals().get("STRICT_SENTENCE_ONLY", True) or ends_strong_sentenceish(remain):
            out.append(remain)
    # Final validation.
    final = []
    for s in out:
        q = text_metrics(s)
        if globals().get("STRICT_SENTENCE_ONLY", True):
            if begins_continuation(s) and final:
                final[-1] = norm_text(final[-1] + " " + s)
                continue
            if not ends_strong_sentenceish(s) and q["word_count"] < 12:
                continue
        if len(s) < 12 and q["domain_heading_hits"] == 0:
            continue
        if q["quality_score"] < MIN_KEEP_SENTENCE_QUALITY - 10 and q["word_count"] < 5:
            continue
        final.append(s)
    return final


In [10]:
# ============================================================
# 8B. DNTC v3 patch: safer reflow + historical Vietnamese sentence splitting
# ============================================================
# This cell overrides a few functions from Cell 8 and appends additional
# conservative correction rules. It must run before Cell 9 processes PDFs.

LETTER_RE_SRC = "A-Za-z\u00C0-\u1EF9\u0110\u0111"
HAN_RE_SRC = "\u3400-\u9FFF"

V3_EXTRA_CORRECTION_RULES = [
    # Missing/garbled opener in the Bai Tu page: ': Doi thanh...' -> 'Trom nghi: Doi thanh...'
    ("v3_trom_nghi_missing_before_doi", "^\\s*[:;]\\s*(?=\u0110\u1eddi\\s+th\u1ea1nh|Doi\\s+thanh|\u0110\u1eddi\\s+thanh)", "Tr\u1ed9m ngh\u0129: "),
    ("v3_trom_nghi_no_diacritic", "\\bTrom\\s+nghi\\s*[:;]", "Tr\u1ed9m ngh\u0129:"),

    # Broken DNTC administrative / title compounds.
    ("v3_tong_tai", "\\bT\u1ed3ng\\s*-?\\s*t\u00e0i\\b|\\bTong\\s*-?\\s*tai\\b", "T\u1ed5ng-t\u00e0i"),
    ("v3_toan_tu", "\\bTo\u1ea3n\\s*-?\\s*tu\\b|\\bToan\\s*-?\\s*tu\\b", "To\u1ea3n-tu"),
    ("v3_quoc_su_quan_dot", "\\bQu\u1ed1c\\s*-?\\s*s\u1eed\\s*[.]\\s*Qu\u00e1n\\b", "Qu\u1ed1c-s\u1eed-qu\u00e1n"),
    ("v3_quoc_su_quan_hyphen", "\\bQu\u1ed1c\\s*-?\\s*s\u1eed\\s*-?\\s*qu\u00e1n\\b", "Qu\u1ed1c-s\u1eed-qu\u00e1n"),
    ("v3_kham_mang", "\\bKh\u00e2m\\s*-?\\s*m\u1ea1ng\\b", "Kh\u00e2m-m\u1ea1ng"),
    ("v3_dai_nam_nhat_thong_chi", "\\b\u0110\u1ea1i\\s*-\\s*Nam\\s*-\\s*Nh\u1ea5t\\s*-\\s*Th\u1ed1ng\\s*-?\\s*Ch[\u00ed\u1ec9i]\b", "\u0110\u1ea1i-Nam-Nh\u1ea5t-Th\u1ed1ng-Ch\u00ed"),

    # Hano-Vietnamese compounds and common OCR substitutions in Bai Tu pages.
    ("v3_thanh_tri", "\\bth\u1ea1nh\\s*-\\s*tr\u1ecb\\b|\\bthanh\\s*-\\s*tri\\b", "th\u1ea1nh-tr\u1ecb"),
    ("v3_xa_thu", "\\bxa\\s+th[\u01a1o]\b", "xa th\u01b0"),
    ("v3_xa_thu_han_spaced", "\\bxa\\s+th\u01b0\\s+\u8eca\\s+\u66f8\\b", "xa th\u01b0 \u8eca\u66f8"),
    ("v3_chuc_phuong", "\\bch\u1ee9c\\s*-\\s*ph\u01b0\u01a1ng\\b", "ch\u1ee9c-ph\u01b0\u01a1ng"),
    ("v3_kinh_vi", "\\bKinh\\s*-\\s*v[\u0129i]\b", "Kinh-v\u0129"),
    ("v3_xuan_thu", "\\bXu\u00e2n\\s*-\\s*thu\\b", "Xu\u00e2n-thu"),
    ("v3_trung_dung", "\\bTrung\\s*-\\s*dung\\b", "Trung-dung"),
    ("v3_nhat_thong_lower", "\\bnh\u1ea5t\\s*-\\s*th\u1ed1ng\\b", "nh\u1ea5t-th\u1ed1ng"),
    ("v3_thuy_tho", "\\bth[\u1ee7u]y\\s+th[\u1ed3o]\b", "th\u1ee7y th\u1ed5"),
    ("v3_an_ngu", "\\b\u1ea7n\\s+ng\u1ee5\\b", "\u1ea9n ng\u1ee5"),
    ("v3_dong_quy", "\\b\u0111\u1ed3ng\\s+qu[\u00edi]\b", "\u0111\u1ed3ng qu\u1ef9"),
    ("v3_tan_trinh", "\\bt\u1ea5n\\s+trinh\\b", "t\u1ea5n tr\u00ecnh"),
    ("v3_can_tau", "\\bc\u1ea7n\\s+t\u1ea5u\\b", "c\u1ea9n t\u1ea5u"),
    ("v3_phap_do", "\\bph\u00e1p\\s+\u0111\u1ed9\\b", "ph\u00e1p \u0111\u1ed9"),
]

# Append only once.
_existing_rule_ids = {r[0] for r in CORRECTION_RULES}
for _rule in V3_EXTRA_CORRECTION_RULES:
    if _rule[0] not in _existing_rule_ids:
        CORRECTION_RULES.append(_rule)

MEANINGFUL_SHORT_LINE_RE = re.compile(
    "^(?:Tr\u1ed9m\\s+ngh\u0129|X\u00e9t\\s+r\u1eb1ng|L\u1eddi\\s+r\u1eb1ng|Nay\\s+k\u00ednh\\s+t\u00e2u|C\u1ea9n\\s+t\u1ea5u|C\u1ea9n\\s+\u00e1n)\\s*[:;]?$",
    re.I,
)

DANGLING_ENDINGS = [
    "nh\u01b0", "\u1edf", "v\u1ec1", "v\u00e0", "l\u00e0", "c\u1ee7a", "\u0111\u01b0\u1ee3c", "r\u1eb1ng",
    "theo", "do", "\u0111\u1ec3", "v\u1edbi", "t\u1eeb", "c\u00e1c", "nh\u1eefng", "n\u01a1i",
    "cho n\u00ean", "b\u1edfi theo", "\u00fd n\u00f3i", "c\u00f3 c\u00e2u:", "cho \u0111\u01b0\u1ee3c:",
]

BROKEN_LINE_END_RE = re.compile(
    "(?:Qu\u1ed1c\\s*-?\\s*s\u1eed\\s*[.]?$|\u0110\u1ea1i\\s*-\\s*Nam\\s*-?$|Nh\u1ea5t\\s*-\\s*Th\u1ed1ng\\s*-?$|Xu\u00e2n\\s*-?$|Kinh\\s*-?$)",
    re.I,
)

BROKEN_LINE_START_RE = re.compile(
    "^(?:Qu\u00e1n\\b|Nam\\b|Ch[\u00ed\u1ec9i]\\b|thu\\b|v[\u0129i]\\b|th\u1ed1ng\\b|m\u1ed9t\\b)",
    re.I,
)


def is_meaningful_short_line(s: str) -> bool:
    return bool(MEANINGFUL_SHORT_LINE_RE.match(norm_text(s)))


def ends_hard_sentence(t: str) -> bool:
    s = norm_text(t)
    return bool(re.search("[.!?\u2026][\"'\u201d\u2019)\\]]*\\s*$", s))


def ends_soft_clause(t: str) -> bool:
    s = norm_text(t)
    return bool(re.search("[:;][\"'\u201d\u2019)\\]]*\\s*$", s))


def ends_dangling(t: str) -> bool:
    s = norm_text(t).lower().strip(" \t\r\n\"'\u201c\u201d\u2018\u2019()[]{}")
    if not s:
        return True
    if s.endswith(":") or s.endswith(";"):
        return True
    return any(s.endswith(x) for x in DANGLING_ENDINGS)


def begins_continuation(t: str) -> bool:
    s = norm_text(t)
    if not s:
        return False
    if re.match("^[\\s:;,\-\u2013\u2014]+", s):
        return True
    s2 = s.lstrip(" \\\"'\u201c\u201d\u2018\u2019([{<\u00ab\u00bb\-\u2013\u2014")
    return bool(s2 and re.match("^[a-z\u00e0-\u1ef9\u0111\u3400-\u9FFF]", s2))

# Override the old broad definition. Colon/semicolon are clauses, not final sentence ends.
ends_strong_sentenceish = ends_hard_sentence

_ORIG_is_line_hard_noise_v3 = is_line_hard_noise

def is_line_hard_noise(s: str) -> bool:
    if is_meaningful_short_line(s):
        return False
    return _ORIG_is_line_hard_noise_v3(s)

_ORIG_filter_line_v3 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    if raw and is_meaningful_short_line(raw):
        meta = {
            "work_id": line.get("work_id"), "pdf_path": line.get("pdf_path"), "page_number": line.get("page_number"),
            "page_idx": line.get("page_idx"), "line_id": line.get("line_global_id"), "source": line.get("source"),
        }
        fixed = apply_corrections(raw, meta, correction_log)
        q = line_quality(fixed, line.get("ocr_conf"))
        kept = dict(line)
        kept.update({
            "text": fixed,
            "line_type": "body",
            "line_quality_score": q["quality_score"],
            "line_vietnamese_ratio": q["vietnamese_ratio"],
            "line_weird_char_ratio": q["weird_char_ratio"],
            "line_ocr_text": "",
            "line_ocr_conf": None,
            "line_ocr_accepted": False,
            "filter_reasons": "kept_meaningful_short_opener",
        })
        return kept, True, ["kept_meaningful_short_opener"]
    return _ORIG_filter_line_v3(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)


def apply_dntc_reflow_corrections(text: str) -> str:
    s = norm_text(text)
    if not s:
        return s
    # Correct phrases that only become visible after joining OCR/PDF lines.
    s = re.sub("\\bQu\u1ed1c\\s*-?\\s*s\u1eed\\s*[.]\\s*Qu\u00e1n\\b", "Qu\u1ed1c-s\u1eed-qu\u00e1n", s, flags=re.I)
    s = re.sub("\\bQu\u1ed1c\\s*-?\\s*s\u1eed\\s*-?\\s*Qu\u00e1n\\b", "Qu\u1ed1c-s\u1eed-qu\u00e1n", s, flags=re.I)
    s = re.sub("\\b\u0110\u1ea1i\\s*-\\s*Nam\\s*-\\s*Nh\u1ea5t\\s*-\\s*Th\u1ed1ng\\s*-?\\s*Ch[\u00ed\u1ec9i]\\b", "\u0110\u1ea1i-Nam-Nh\u1ea5t-Th\u1ed1ng-Ch\u00ed", s, flags=re.I)
    s = re.sub("\\bT\u1ed5ng\\s*-\\s*t\u00e0i\\b", "T\u1ed5ng-t\u00e0i", s, flags=re.I)
    s = re.sub("\\bTo\u1ea3n\\s*-\\s*tu\\b", "To\u1ea3n-tu", s, flags=re.I)
    s = re.sub("\\bKh\u00e2m\\s*-\\s*m\u1ea1ng\\b", "Kh\u00e2m-m\u1ea1ng", s, flags=re.I)
    s = re.sub("\\bth\u1ea1nh\\s*-\\s*tr\u1ecb\\b", "th\u1ea1nh-tr\u1ecb", s, flags=re.I)
    s = re.sub("\\bKinh\\s*-\\s*v[\u0129i]\\b", "Kinh-v\u0129", s, flags=re.I)
    s = re.sub("\\bXu\u00e2n\\s*-\\s*thu\\b", "Xu\u00e2n-thu", s, flags=re.I)
    s = candidate_correction_only(s)
    s = re.sub("\\s+([,.;:!?])", "\\1", s)
    s = re.sub("([,.;:!?])(?=\\S)", "\\1 ", s)
    s = re.sub("\\s+", " ", s).strip()
    return s


def should_merge_lines(prev: dict, cur: dict, med_h: float, page_rect) -> bool:
    pt = norm_text(prev.get("text", ""))
    ct = norm_text(cur.get("text", ""))
    if not pt or not ct:
        return False
    if prev.get("line_type") == "heading" or cur.get("line_type") == "heading":
        return False
    if prev.get("is_footnote") != cur.get("is_footnote"):
        return False
    # Footnote blocks at bottom should remain separate; inline notes stay in paragraph by geometry.
    if re.match("^\\(?\\d+\\)|^\\d+[.)]", ct) and (cur.get("bbox", [0, 0, 0, 0])[1] > getattr(page_rect, "height", 99999) * 0.72):
        return False
    if BROKEN_LINE_END_RE.search(pt) or BROKEN_LINE_START_RE.match(ct):
        return True
    if ends_soft_clause(pt) or ends_dangling(pt):
        return True
    if not ends_hard_sentence(pt):
        return True
    if begins_continuation(ct):
        return True
    try:
        gap = cur["bbox"][1] - prev["bbox"][3]
        indent_delta = cur["bbox"][0] - prev["bbox"][0]
        if gap < med_h * 1.15 and abs(indent_delta) < med_h * 2.5:
            # Small line gap inside same text block: keep together unless previous sentence is clearly complete
            # and current line looks like a real new sentence opener.
            if not (ends_hard_sentence(pt) and re.match("^[A-Z\u00c0-\u1ef9\u0110]", ct)):
                return True
    except Exception:
        pass
    return False


def should_new_paragraph(prev: dict, cur: dict, med_h: float, page_rect) -> bool:
    if prev.get("line_type") == "heading" or cur.get("line_type") == "heading":
        return True
    if prev.get("is_footnote") != cur.get("is_footnote"):
        return True
    if should_merge_lines(prev, cur, med_h, page_rect):
        return False
    pt, ct = norm_text(prev.get("text", "")), norm_text(cur.get("text", ""))
    try:
        gap = cur["bbox"][1] - prev["bbox"][3]
        indent_delta = cur["bbox"][0] - prev["bbox"][0]
        if gap > med_h * 2.35 and ends_hard_sentence(pt) and not begins_continuation(ct):
            return True
        if indent_delta > med_h * 3.0 and len(pt) > 35 and ends_hard_sentence(pt) and not begins_continuation(ct):
            return True
    except Exception:
        pass
    return False


def join_paragraph_lines(lines: list) -> str:
    parts = []
    for r in lines:
        t = norm_text(r.get("text", ""))
        if not t:
            continue
        if not parts:
            parts.append(t)
            continue
        prev = parts[-1]
        # True word split: 'huy-' + 'en' -> 'huyen'. Keep historical hyphenated compounds otherwise.
        if re.search("[a-z\u00e0-\u1ef9\u0111]-$", prev) and re.match("^[a-z\u00e0-\u1ef9\u0111]", t):
            parts[-1] = prev[:-1] + t
        else:
            parts.append(t)
    return apply_dntc_reflow_corrections(" ".join(parts))


def looks_like_sentence_fragment(s: str) -> bool:
    s = norm_text(s)
    if not s:
        return True
    q = text_metrics(s)
    if re.match("^[\\s:;,\-\u2013\u2014]+", s):
        return True
    if ends_dangling(s):
        return True
    if begins_continuation(s) and q["word_count"] < 16:
        return True
    if not ends_hard_sentence(s) and q["word_count"] < 14:
        return True
    if CLEAR_JUNK_RE.search(s) or is_ascii_short_junk_line(s):
        return True
    return False


def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    text = apply_dntc_reflow_corrections(paragraph_text)
    if paragraph_type == "heading" or not text:
        return []
    protected, repl = protect_sentence_abbrevs(text)
    out, buf = [], []
    depth = 0
    for i, ch in enumerate(protected):
        buf.append(ch)
        if ch in "([{" or ch == "\u00ab":
            depth += 1
        elif ch in ")]}" or ch == "\u00bb":
            depth = max(0, depth - 1)
        if ch not in ".?!\u2026":
            continue
        tail = "".join(buf).strip()
        if not tail:
            continue
        if depth > 0:
            continue
        if re.search("\\(\\s*\\d+\\s*\\)\\s*$", tail):
            continue
        # Do not split if the next visible token is lowercase, Han, punctuation, or a continuation marker.
        rest = protected[i + 1:]
        m = re.search("\\S", rest)
        if m:
            nchar = rest[m.start()]
            if re.match("[a-z\u00e0-\u1ef9\u0111\u3400-\u9FFF:;,\-\u2013\u2014]", nchar):
                continue
        sent = apply_dntc_reflow_corrections(unprotect_sentence_abbrevs(tail, repl))
        if sent:
            out.append(sent)
        buf = []
    remain = apply_dntc_reflow_corrections(unprotect_sentence_abbrevs("".join(buf).strip(), repl))
    if remain:
        if out and (begins_continuation(remain) or ends_dangling(out[-1]) or looks_like_sentence_fragment(remain)):
            out[-1] = apply_dntc_reflow_corrections(out[-1] + " " + remain)
        elif not globals().get("STRICT_SENTENCE_ONLY", True) or not looks_like_sentence_fragment(remain):
            out.append(remain)
    final = []
    for s in out:
        s = apply_dntc_reflow_corrections(s)
        q = text_metrics(s)
        if final and begins_continuation(s):
            final[-1] = apply_dntc_reflow_corrections(final[-1] + " " + s)
            continue
        if globals().get("STRICT_SENTENCE_ONLY", True) and looks_like_sentence_fragment(s):
            # Do not export orphan fragments to final_sentences_only.
            continue
        if len(s) < 12 and q["domain_heading_hits"] == 0:
            continue
        if q["quality_score"] < MIN_KEEP_SENTENCE_QUALITY - 10 and q["word_count"] < 5:
            continue
        final.append(s)
    return final

print("DNTC v3 patch loaded: conservative reflow, opener preservation, no semicolon split, strict fragment blocking.")
# --- v3.1 safety override: paragraph-level corrections must not reuse heading rules ---
# Some heading rules intentionally uppercase title lines. For body paragraphs they can damage
# compounds such as Dai-Nam-Nhat-Thong-Chi, so post-join correction uses a safe local list.
_CORRECTION_RULES_SANITIZED = []
for _rid, _pat, _repl in CORRECTION_RULES:
    if isinstance(_pat, str):
        _pat = _pat.replace("\x08", r"\b")
    _CORRECTION_RULES_SANITIZED.append((_rid, _pat, _repl))
CORRECTION_RULES[:] = _CORRECTION_RULES_SANITIZED

_WB = r"\b"
DNTC_V3_SAFE_POSTJOIN_RULES = [
    (_WB + "Trom\\s+nghi\\s*[:;]", "Tr\u1ed9m ngh\u0129:"),
    ("^\\s*[:;]\\s*(?=\\u0110\\u1eddi\\s+th\\u1ea1nh|Doi\\s+thanh|\\u0110\\u1eddi\\s+thanh)", "Tr\u1ed9m ngh\u0129: "),
    (_WB + "T\\u1ed3ng\\s*-?\\s*t\\u00e0i" + _WB, "T\u1ed5ng-t\u00e0i"),
    (_WB + "Tong\\s*-?\\s*tai" + _WB, "T\u1ed5ng-t\u00e0i"),
    (_WB + "To\\u1ea3n\\s*-?\\s*tu" + _WB, "To\u1ea3n-tu"),
    (_WB + "Qu\\u1ed1c\\s*-?\\s*s\\u1eed\\s*[.]\\s*Qu\\u00e1n" + _WB, "Qu\u1ed1c-s\u1eed-qu\u00e1n"),
    (_WB + "Qu\\u1ed1c\\s*-?\\s*s\\u1eed\\s*-?\\s*Qu\\u00e1n" + _WB, "Qu\u1ed1c-s\u1eed-qu\u00e1n"),
    (_WB + "Kh\\u00e2m\\s*-?\\s*m\\u1ea1ng" + _WB, "Kh\u00e2m-m\u1ea1ng"),
    (_WB + "\\u0110\\u1ea1i\\s*-\\s*Nam\\s*-\\s*Nh\\u1ea5t\\s*-\\s*Th\\u1ed1ng\\s*-?\\s*Ch[\\u00ed\\u1ec9i]" + _WB, "\u0110\u1ea1i-Nam-Nh\u1ea5t-Th\u1ed1ng-Ch\u00ed"),
    (_WB + "th\\u1ea1nh\\s*-\\s*tr\\u1ecb" + _WB, "th\u1ea1nh-tr\u1ecb"),
    (_WB + "thanh\\s*-\\s*tri" + _WB, "th\u1ea1nh-tr\u1ecb"),
    (_WB + "xa\\s+th[\\u01a1o]" + _WB, "xa th\u01b0"),
    (_WB + "xa\\s+th\\u01b0\\s+\\u8eca\\s+\\u66f8" + _WB, "xa th\u01b0 \u8eca\u66f8"),
    (_WB + "ch\\u1ee9c\\s*-\\s*ph\\u01b0\\u01a1ng" + _WB, "ch\u1ee9c-ph\u01b0\u01a1ng"),
    (_WB + "Kinh\\s*-\\s*v[\\u0129i]" + _WB, "Kinh-v\u0129"),
    (_WB + "Xu\\u00e2n\\s*-\\s*thu" + _WB, "Xu\u00e2n-thu"),
    (_WB + "Trung\\s*-\\s*dung" + _WB, "Trung-dung"),
    (_WB + "nh\\u1ea5t\\s*-\\s*th\\u1ed1ng" + _WB, "nh\u1ea5t-th\u1ed1ng"),
    (_WB + "th[\\u1ee7u]y\\s+th[\\u1ed3o]" + _WB, "th\u1ee7y th\u1ed5"),
    (_WB + "\\u1ea7n\\s+ng\\u1ee5" + _WB, "\u1ea9n ng\u1ee5"),
    (_WB + "\\u0111\\u1ed3ng\\s+qu[\\u00edi]" + _WB, "\u0111\u1ed3ng qu\u1ef9"),
    (_WB + "c\\u1ea7n\\s+t\\u1ea5u" + _WB, "c\u1ea9n t\u1ea5u"),
]


def apply_dntc_reflow_corrections(text: str) -> str:
    s = norm_text(text)
    if not s:
        return s
    for _pat, _repl in DNTC_V3_SAFE_POSTJOIN_RULES:
        s = re.sub(_pat, _repl, s, flags=re.I)
    # Cleanup common OCR spacing around Han tokens, parentheses, and punctuation.
    s = re.sub(r"([\u3400-\u9FFF])\s+([\u3400-\u9FFF])", r"\1\2", s)
    s = re.sub(r"\(\s+", "(", s)
    s = re.sub(r"\s+\)", ")", s)
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,.;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

print("DNTC v3.1 safety override loaded: safe post-join corrections active.")
# --- v3.2 targeted Bai Tu paragraph repairs ---
# These rules target a recurring line-break/OCR pattern in the prefatory Bai Tu pages.
DNTC_V3_SAFE_POSTJOIN_RULES.extend([
    (r"([.!?])\s*[:;]\s*(?=\u0110\u1eddi\s+th\u1ea1nh|Doi\s+thanh|\u0110\u1eddi\s+thanh)", "\\1 Tr\u1ed9m ngh\u0129: "),
    (r"\b\u0110\u1ea1i-Nam-nh\u1ea5t-th\u1ed1ng-Ch[\u00ed\u1ec9i]\b", "\u0110\u1ea1i-Nam-Nh\u1ea5t-Th\u1ed1ng-Ch\u00ed"),
    (r"\bXu\u00e2n\s*-\s*\(4\)", "Xu\u00e2n-thu nh\u1ea5t-th\u1ed1ng (4)"),
    (r"\b\u0110[\u1ec1\u1ec3e]\s+thu\s+nh\u1ea5t-th\u1ed1ng\s+cho\s+\u0111\u01b0\u1ee3c\b", "\u0110\u1ec3 cho \u0111\u01b0\u1ee3c"),
])
print("DNTC v3.2 targeted Bai Tu repairs loaded.")




DNTC v3 patch loaded: conservative reflow, opener preservation, no semicolon split, strict fragment blocking.
DNTC v3.1 safety override loaded: safe post-join corrections active.
DNTC v3.2 targeted Bai Tu repairs loaded.


<>:95: SyntaxWarning: invalid escape sequence '\-'
<>:97: SyntaxWarning: invalid escape sequence '\-'
<>:234: SyntaxWarning: invalid escape sequence '\-'
<>:274: SyntaxWarning: invalid escape sequence '\-'
<>:95: SyntaxWarning: invalid escape sequence '\-'
<>:97: SyntaxWarning: invalid escape sequence '\-'
<>:234: SyntaxWarning: invalid escape sequence '\-'
<>:274: SyntaxWarning: invalid escape sequence '\-'
/tmp/ipykernel_39/687819039.py:95: SyntaxWarning: invalid escape sequence '\-'
  if re.match("^[\\s:;,\-\u2013\u2014]+", s):
/tmp/ipykernel_39/687819039.py:97: SyntaxWarning: invalid escape sequence '\-'
  s2 = s.lstrip(" \\\"'\u201c\u201d\u2018\u2019([{<\u00ab\u00bb\-\u2013\u2014")
/tmp/ipykernel_39/687819039.py:234: SyntaxWarning: invalid escape sequence '\-'
  if re.match("^[\\s:;,\-\u2013\u2014]+", s):
/tmp/ipykernel_39/687819039.py:274: SyntaxWarning: invalid escape sequence '\-'
  if re.match("[a-z\u00e0-\u1ef9\u0111\u3400-\u9FFF:;,\-\u2013\u2014]", nchar):


In [11]:
# ============================================================
# 8C. DNTC v4 patch: old-scan OCR numeric/domain cleanup
# ============================================================
# This cell is intentionally placed after v3 and before processing.
# It focuses on old 1960s scanned volumes such as 14_15 where PDF text layer is empty
# and Tesseract is used. The rules are conservative and context-bound.

# Use a little more resolution for old Vietnamese typefaces. Slower, but safer for full quality.
try:
    PAGE_OCR_DPI = max(int(PAGE_OCR_DPI), 320)
except Exception:
    PAGE_OCR_DPI = 320
OCR_DPI = PAGE_OCR_DPI

V4_EXTRA_CORRECTION_RULES = [
    # Remove isolated OCR/copyright-like markers explicitly requested by reviewer.
    ("v4_drop_rc_marker_inline", r"\s*\(([RC])\)\s*", " "),

    # Old scan headings and administrative parentheticals.
    ("v4_luong_son_dot", r"\bLƯƠNG\s*[.]\s*SƠN\b|\bLUONG\s*[.]\s*SON\b", "LƯƠNG-SƠN"),
    ("v4_thanh_chuong_ascii", r"\bTHANH\s*-\s*CHU['’]?ONG\b|\bTHANH\s*-\s*CHUONG\b", "THANH-CHƯƠNG"),
    ("v4_huyen_heading_ascii", r"^\s*[|]?\s*HUYEN\b", "HUYỆN"),
    ("v4_do_phu_kiem_ly", r"\((?:do|đo)\s+ph[ảa]\s+ki[ée]m\s*-\s*l[yý]\)", "(do phủ kiêm-lý)"),
    ("v4_do_phu_thong_hat", r"\((?:do|đo)\s+(?:ph[ảa]|phi)\s+th[oôd]ng\s*-\s*(?:h[ạa]?t|hgt)\)", "(do phủ thống-hạt)"),
    ("v4_thong_hat_broken", r"\bthống\s*[.]\s*h[ạa]t\b|\bthống\s*-\s*hgt\b|\bth[oôd]ng\s*-\s*hgt\b", "thống-hạt"),
    ("v4_kiem_ly", r"\bki[ée]m\s*-\s*l[yý]\b", "kiêm-lý"),

    # Common old-scan OCR substitutions in Nghệ An / Thanh Hóa volumes.
    ("v4_tong", r"\btồng\b", "tổng"),
    ("v4_giap", r"\bgiấp\b", "giáp"),
    ("v4_dam_typo", r"\bđặm\b", "dặm"),
    ("v4_nam_reign", r"\b(?:Nim|Nam)\s+(?=(?:Minh|Tự|Thành|Gia|Đồng|Thiệu|Kiến|Duy)\s*-?\s*[A-ZÀ-ỸĐ])", "Năm "),
    ("v4_huyen_ly", r"\bHuyện\s*[.]\s*l[yỵi]\b", "Huyện-lỵ"),
    ("v4_do_luong", r"\bĐô\s*[.]\s*Lương\b", "Đô-Lương"),
    ("v4_anh_do", r"\bAnh\s*[.]\s*Đô\b", "Anh-Đô"),
    ("v4_dong_khanh", r"\bĐồng\s*[.]\s*Khánh\b", "Đồng-Khánh"),
    ("v4_thanh_thai", r"\bThành\s*[.]\s*Thái\b", "Thành-Thái"),
    ("v4_nnam", r"\bN+nam\b", "Nam"),
    ("v4_la_nam", r"\bLaNam\b", "La-Nam"),
    ("v4_phia_dong_nam_phu", r"\bphía\s+đông\s+nam\s+ph[úu]\b", "phía đông nam phủ"),
    ("v4_doi_context", r"\bđồi(?=\s+(?:là|làm|tên|thuộc|lệ|sang|gọi|huyện|phủ))", "đổi"),
    ("v4_da_bo_di", r"\bđã\s+bỏ\s+d[iì]\b", "đã bỏ đi"),
    ("v4_tri_huyen", r"\bTri\s*-\s*huyện\b", "Tri huyện"),

    # Tax / census units and mojibake in numeric pages.
    ("v4_thue_mojibake", r"(?<!\w)thu[€éêể](?!\w)", "thuế"),
    ("v4_dien_tho", r"\bđiền\s+th[ồo]\b|\bdin\s+thd\b", "điền thổ"),
    ("v4_bac_thue", r"\bbac\s+thu[eéế]\b", "bạc thuế"),
    ("v4_mau_digit", r"\bm4u\b", "mẫu"),
    ("v4_cong", r"\bc[ée]ng\b", "cộng"),
    ("v4_moi_dinh", r"\bméi\s*dinh\b|\bméidinh\b", "mới định"),
    ("v4_thang_unit", r"\bthing\b", "thăng"),
    ("v4_hap_unit", r"\bhap\b", "hạp"),
    ("v4_thuoc_unit", r"\bthirgc\b|\bthugc\b", "thược"),
    ("v4_lai_phu_nap", r"\bLai\s+phy\s+nap\b", "Lại phụ nạp"),
]

_existing_rule_ids = {r[0] for r in CORRECTION_RULES}
for _rule in V4_EXTRA_CORRECTION_RULES:
    if _rule[0] not in _existing_rule_ids:
        CORRECTION_RULES.append(_rule)

DNTC_V4_SAFE_POSTJOIN_RULES = [
    # Remove reviewer-reported isolated markers.
    (r"\s*\(([RC])\)\s*", " "),

    # Clean page/line OCR debris.
    (r"^\s*[|]\s*", ""),
    (r"\s+[|]\s*", " "),
    (r"\s+_\s*", " "),
    (r"\s+'(?=\s*[ĐA-ZÀ-Ỹđa-zà-ỹ])", " "),
    (r"\s+", " "),

    # Headings/domain forms.
    (r"\bHUYEN\b", "HUYỆN"),
    (r"\bLƯƠNG\s*[.]\s*SƠN\b|\bLUONG\s*[.]\s*SON\b", "LƯƠNG-SƠN"),
    (r"\bTHANH\s*-\s*CHU['’]?ONG\b|\bTHANH\s*-\s*CHUONG\b", "THANH-CHƯƠNG"),
    (r"\((?:do|đo)\s+ph[ảa]\s+ki[ée]m\s*-\s*l[yý]\)", "(do phủ kiêm-lý)"),
    (r"\((?:do|đo)\s+(?:ph[ảa]|phi)\s+th[oôd]ng\s*-\s*(?:h[ạa]?t|hgt)\)", "(do phủ thống-hạt)"),
    (r"\bthống\s*[.]\s*h[ạa]t\b|\bthống\s*-\s*hgt\b|\bth[oôd]ng\s*-\s*hgt\b", "thống-hạt"),
    (r"\bki[ée]m\s*-\s*l[yý]\b", "kiêm-lý"),

    # Common OCR words.
    (r"\btồng\b", "tổng"),
    (r"\bgiấp\b", "giáp"),
    (r"\bđặm\b", "dặm"),
    (r"\b(?:Nim|Nam)\s+(?=(?:Minh|Tự|Thành|Gia|Đồng|Thiệu|Kiến|Duy)\s*-?\s*[A-ZÀ-ỸĐ])", "Năm "),
    (r"\bHuyện\s*[.]\s*l[yỵi]\b", "Huyện-lỵ"),
    (r"\bĐô\s*[.]\s*Lương\b", "Đô-Lương"),
    (r"\bAnh\s*[.]\s*Đô\b", "Anh-Đô"),
    (r"\bĐồng\s*[.]\s*Khánh\b", "Đồng-Khánh"),
    (r"\bThành\s*[.]\s*Thái\b", "Thành-Thái"),
    (r"\bN+nam\b", "Nam"),
    (r"\bLaNam\b", "La-Nam"),
    (r"\bphía\s+đông\s+nam\s+ph[úu]\b", "phía đông nam phủ"),
    (r"\bđồi(?=\s+(?:là|làm|tên|thuộc|lệ|sang|gọi|huyện|phủ))", "đổi"),
    (r"\bđã\s+bỏ\s+d[iì]\b", "đã bỏ đi"),
    (r"\bTri\s*-\s*huyện\b", "Tri huyện"),

    # Specific geography line break: "Đông. _ giáp..." is one clause, not a sentence.
    (r"\b(Đông|Tây|Nam|Bắc)\s*[.]\s*[_']?\s*(?=gi[áa]p\b)", r"\1 "),
    (r"\b(cách nhau\s+\d+\s+dặm)\s*[.]\s+(nam\s+bắc)\b", r"\1, \2"),

    # Tax / census pages.
    (r"(?<!\w)thu[€éêể](?!\w)", "thuế"),
    (r"\bđiền\s+th[ồo]\b|\bdin\s+thd\b", "điền thổ"),
    (r"\bbac\s+thu[eéế]\b", "bạc thuế"),
    (r"\bm4u\b", "mẫu"),
    (r"\bc[ée]ng\b", "cộng"),
    (r"\bméi\s*dinh\b|\bméidinh\b", "mới định"),
    (r"\bthing\b", "thăng"),
    (r"\bhap\b", "hạp"),
    (r"\bthirgc\b|\bthugc\b", "thược"),
    (r"\bLai\s+phy\s+nap\b", "Lại phụ nạp"),
]


def fix_old_scan_numeric_ocr(s: str) -> str:
    """Contextual numeric cleanup for old Tesseract OCR.
    It only touches number-like tokens near units/reign-year contexts to avoid
    rewriting normal Vietnamese words.
    """
    if not s:
        return s

    # Common year / reign OCR.
    s = re.sub(r"\br8so\b", "1850", s, flags=re.I)
    s = re.sub(r"\br886\b", "1886", s, flags=re.I)
    s = re.sub(r"\b18g8\b", "1898", s, flags=re.I)
    s = re.sub(r"\br1oo\b", "1100", s, flags=re.I)
    s = re.sub(r"\b(th[ứưửu]\s+)ro\b", r"\g<1>10", s, flags=re.I)
    s = re.sub(r"\b(th[ứưửu]\s+)rz\b", r"\g<1>12", s, flags=re.I)
    s = re.sub(r"\b(th[ứưửu]\s+)a1\b", r"\g<1>21", s, flags=re.I)

    # Standalone OCR for 9/5 in month expressions.
    s = re.sub(r"\b(mồng|tháng)\s+o\b", r"\1 9", s, flags=re.I)
    s = re.sub(r"\b(mồng|tháng)\s+s\b", r"\1 5", s, flags=re.I)

    # Leading o before a digit in distance is usually 9: o4 dặm -> 94 dặm.
    s = re.sub(r"\bo(?=\d+\s+dặm\b)", "9", s, flags=re.I)

    # Standalone 's' before administrative/statistical units is usually 5.
    s = re.sub(r"\bs(?=\s+(?:tổng|sào|thôn|xã|mẫu|người|lạng|thước|đồng|tiền|quan|phần|huyện)\b)", "5", s, flags=re.I)

    # Confusable trailing letters inside numeric spans before units.
    unit = r"(?:dặm|tổng|sào|thôn|xã|mẫu|người|lạng|thước|đồng|tiền|quan|phần|huyện|đ|đồng)"
    s = re.sub(r"(?<=\d)s(?=\s+" + unit + r"\b)", "5", s, flags=re.I)
    s = re.sub(r"(?<=\d)o(?=\s+" + unit + r"\b)", "0", s, flags=re.I)
    s = re.sub(r"(?<=\d)g(?=\d|\s*" + unit + r"\b)", "9", s, flags=re.I)

    # General number-like tokens containing OCR letters and at least one digit,
    # only if followed by a known unit or currency marker.
    def _fix_token(m):
        tok = m.group(1)
        table = str.maketrans({
            "o": "0", "O": "0",
            "s": "5", "S": "5",
            "g": "9", "q": "9",
            "r": "1", "l": "1", "I": "1",
            "z": "2", "Z": "2",
        })
        return tok.translate(table)

    s = re.sub(r"\b([0-9osgqrIlzZ]{2,})(?=\s*(?:đ|đồng|người|mẫu|dặm|quan|lạng|thăng|hạp|thược|sào|thước|loát)\b)", _fix_token, s, flags=re.I)

    # Add missing space: 4tổng -> 4 tổng.
    s = re.sub(r"(\d)(?=(?:tổng|xã|thôn|mẫu|sào|thước|dặm)\b)", r"\1 ", s, flags=re.I)

    # OCR sometimes adds an extra letter to "2 huyện": "2a huyện".
    s = re.sub(r"\b2a\s+huyện\b", "2 huyện", s, flags=re.I)

    # Merge thousands / decimal groups wrongly split by sentence punctuation.
    s = re.sub(r"\b(\d{1,3})\.\s+(\d{3})(?=\s+(?:người|mẫu|quan|lạng|đồng|đ|thăng|hạp|thược|sào|thước|loát)\b)", r"\1.\2", s)
    s = re.sub(r"\b(\d{1,3})\s*,\s+(\d{3})(?=\s*(?:người|mẫu|quan|lạng|đồng|đ|thăng|hạp|thược|sào|thước|loát)\b)", r"\1,\2", s)

    return s


_ORIG_apply_dntc_reflow_corrections_v4 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v4(text)
    if not s:
        return s
    for _pat, _repl in DNTC_V4_SAFE_POSTJOIN_RULES:
        s = re.sub(_pat, _repl, s, flags=re.I)
    s = fix_old_scan_numeric_ocr(s)

    # Some rules become active only after numeric OCR has been repaired.
    s = re.sub(r"\b(cách nhau\s+\d+\s+dặm)\s*[.]\s+(nam\s+bắc)\b", r"\1, \2", s, flags=re.I)

    # Cleanup spacing around punctuation and hyphenated compounds after numeric/domain rules.
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    # Do not split numeric groups like 37.996 or 44.237.
    s = re.sub(r"([,;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(?<!\d)\.(?=\S)", ". ", s)
    s = re.sub(r"(?<=\d)\.(?=[^\d\s])", ". ", s)
    s = re.sub(r"\s*-\s*", "-", s)
    # Restore spaces around sentence-level dashes if any were over-normalized is intentionally avoided;
    # DNTC old text uses many compound hyphens.
    s = re.sub(r"\s+", " ", s).strip()
    return s


_ORIG_is_heading_text_v4 = is_heading_text

def is_heading_text(s: str) -> bool:
    s0 = apply_dntc_reflow_corrections(norm_text(s)).strip(" |")
    if _ORIG_is_heading_text_v4(s0):
        return True
    su = s0.upper()
    # Old scan subheadings often contain a lowercase explanatory parenthetical,
    # so uppercase ratio alone misses them.
    if re.match(r"^HUYỆN\s+[A-ZÀ-ỸĐ0-9 .'\-]+(?:\s*\(|$)", su):
        return True
    if re.match(r"^(PHỦ|TỔNG|XÃ|THÀNH|NÚI|SÔNG|ĐỀN|CHÙA|MIẾU)\s+[A-ZÀ-ỸĐ0-9 .'\-]+(?:\s*\(|$)", su) and len(s0) <= 120:
        return True
    return False


_ORIG_is_line_hard_noise_v4 = is_line_hard_noise

def is_line_hard_noise(s: str) -> bool:
    s0 = norm_text(s)
    if re.fullmatch(r"\(?[RC]\)?", s0.strip(), flags=re.I):
        return True
    return _ORIG_is_line_hard_noise_v4(s)


_ORIG_filter_line_v4 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    kept, ok, reasons = _ORIG_filter_line_v4(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if not ok or kept is None:
        return kept, ok, reasons
    fixed = apply_dntc_reflow_corrections(kept.get("text", ""))
    if not fixed or re.fullmatch(r"\(?[RC]\)?", fixed.strip(), flags=re.I):
        return None, False, (reasons or []) + ["drop_rc_marker"]
    kept["text"] = fixed
    q = line_quality(fixed, kept.get("ocr_conf"))
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    if is_heading_text(fixed):
        kept["line_type"] = "heading"
    return kept, True, reasons


# Continuation should treat OCR debris such as "_ giáp" or "' Đây" as a continuation.
def begins_continuation(t: str) -> bool:
    s = norm_text(t)
    if not s:
        return False
    if re.match(r"^[\s:;,._'\"|\\/\-\–\—]+", s):
        return True
    s2 = s.lstrip(" \"'“”‘’([{<«»|-–—_.")
    return bool(s2 and re.match(r"^[a-zà-ỹđ\u3400-\u9FFF]", s2))


print("DNTC v4 patch loaded: old-scan numeric cleanup, HUYỆN subheading detection, (R)/(C) removal, OCR_DPI=", PAGE_OCR_DPI)

DNTC v4 patch loaded: old-scan numeric cleanup, HUYỆN subheading detection, (R)/(C) removal, OCR_DPI= 320


In [12]:

# ============================================================
# 8D. DNTC v5 patch: preserve short meaningful lines + stronger sentence boundaries
# ============================================================
# This patch fixes two classes observed in 01.pdf page 52-53:
# - short meaningful lines such as "bệ có 5 cấp." were dropped as low quality;
# - structural starts like "Từng thứ ba:" and cross-page continuations must be handled.

MEANINGFUL_SHORT_BODY_RE = re.compile(
    r"\b(?:bệ|cấp|án|đàn|móng|trụ|cột|trần\s+thiết|đại\s*-?\s*thứ|"
    r"lò|hầm|gạch|thước|tấc|phân|trượng|mặt|tường|cửa|hàng|lọng|tàn|"
    r"phía|đông|tây|nam|bắc|miếu|điện|thần|khố|trù)\b",
    re.I,
)
OPEN_PAREN_CONTEXT_RE = re.compile(r"^\(?\s*(?:nguyên|xem|tục|nay|cũ|tức|theo)\b", re.I)
STRUCTURAL_SENTENCE_START_RE = re.compile(
    r"^(?:Từng\s+thứ\s+(?:nhất|nhì|ba|tư|bốn|năm|\d+)|"
    r"Đàn\s+(?:vuông|tròn|chế)|Ba\s+từng|Bốn\s+mặt|Ở\s+góc|Phía\s+(?:tả|hữu|đông|tây|nam|bắc)|"
    r"Án\s+(?:tả|hữu|chính)|Hữu\s+(?:nhất|nhị|tam|tứ|tử)|Tả\s+(?:nhất|nhị|tam|tứ|tử)|"
    r"Cẩn\s+án|Lại\s+xét|Năm\s+[A-ZÀ-ỸĐ])\b",
    re.I,
)

_ORIG_filter_line_v5 = filter_line

def _keep_short_meaningful_line(raw_line: dict, correction_log: list):
    raw = norm_text(raw_line.get("raw_text") or raw_line.get("text") or "")
    if not raw:
        return None, False, []
    fixed = apply_corrections(raw, {
        "work_id": raw_line.get("work_id"), "pdf_path": raw_line.get("pdf_path"),
        "page_number": raw_line.get("page_number"), "page_idx": raw_line.get("page_idx"),
        "line_id": raw_line.get("line_global_id"), "source": raw_line.get("source"),
    }, correction_log)
    fixed = apply_dntc_reflow_corrections(fixed)
    q = line_quality(fixed, raw_line.get("ocr_conf"))
    if q["weird_char_ratio"] > 0.04 or q["letter_count"] < 4:
        return None, False, []
    if CLEAR_JUNK_RE.search(fixed) or LIBRARY_NOISE_RE.search(fixed) or is_ascii_short_junk_line(fixed):
        return None, False, []
    looks_meaningful = bool(MEANINGFUL_SHORT_BODY_RE.search(fixed) or OPEN_PAREN_CONTEXT_RE.search(fixed))
    # Keep only short-ish fragments that are semantically useful in layout descriptions.
    if looks_meaningful and q["quality_score"] >= 22.0 and len(fixed) <= 90:
        line_type = "heading" if is_heading_text(fixed) else "body"
        kept = dict(raw_line)
        kept.update({
            "text": fixed,
            "line_type": line_type,
            "line_quality_score": q["quality_score"],
            "line_vietnamese_ratio": q["vietnamese_ratio"],
            "line_weird_char_ratio": q["weird_char_ratio"],
            "line_ocr_text": "",
            "line_ocr_conf": None,
            "line_ocr_accepted": False,
            "filter_reasons": "kept_short_meaningful_body_line_v5",
        })
        return kept, True, ["kept_short_meaningful_body_line_v5"]
    return None, False, []


def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    kept, ok, reasons = _ORIG_filter_line_v5(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if ok and kept is not None:
        return kept, ok, reasons
    # Recover short but meaningful lines which the old score-only gate dropped.
    kept2, ok2, reasons2 = _keep_short_meaningful_line(line, correction_log)
    if ok2:
        return kept2, True, reasons2
    return kept, ok, reasons


_ORIG_apply_dntc_reflow_corrections_v5 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v5(text)
    if not s:
        return s
    # If a short dropped/missing line caused "..., mỗi Từng thứ ba", remove dangling "mỗi" and start a new sentence.
    s = re.sub(r"(bệ\s+đi\s+ra),\s*mỗi\s+(?=Từng\s+thứ\s+(?:nhất|nhì|ba|tư|bốn|năm|\d+)\b)", r"\1. ", s, flags=re.I)
    # Layout enumerators are sentence starts; ensure a boundary before them if previous clause already ended.
    s = re.sub(r"([.!?])\s+(?=(?:Từng\s+thứ|Đàn\s+vuông|Ba\s+từng|Ở\s+góc|Phía\s+(?:tả|hữu)|Án\s+(?:tả|hữu)|Cẩn\s+án)\b)", r"\1 ", s, flags=re.I)
    # Normalize spacing one more time.
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(?<!\d)\.(?=\S)", ". ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


_ORIG_split_sentences_vietnamese_v5 = split_sentences_vietnamese

def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    text = apply_dntc_reflow_corrections(paragraph_text)
    if paragraph_type == "heading" or not text:
        return []
    # Add a virtual sentence boundary before structural starts when a page/layout join removed it.
    text = re.sub(r"(?<=[.!?])\s+(?=(?:Từng\s+thứ|Đàn\s+vuông|Ba\s+từng|Ở\s+góc|Phía\s+(?:tả|hữu)|Án\s+(?:tả|hữu)|Cẩn\s+án)\b)", "\n", text, flags=re.I)
    parts = [p.strip() for p in text.split("\n") if p.strip()]
    out = []
    for part in parts:
        out.extend(_ORIG_split_sentences_vietnamese_v5(part, paragraph_type))
    # Do not merge independent structural sentences back into previous item.
    final = []
    for s in out:
        s = apply_dntc_reflow_corrections(s)
        if not s:
            continue
        if final and begins_continuation(s) and not STRUCTURAL_SENTENCE_START_RE.match(s):
            final[-1] = apply_dntc_reflow_corrections(final[-1] + " " + s)
        else:
            final.append(s)
    return final

print("DNTC v5 pre-process patch loaded: short meaningful lines preserved, structural sentence starts protected.")


DNTC v5 pre-process patch loaded: short meaningful lines preserved, structural sentence starts protected.


In [13]:

# ============================================================
# 8E. DNTC v6 patch: strict heading classifier + strict sentence-only export
# ============================================================
# v5 fixed several cross-page stitches, but review of the new output showed a new
# root cause: heading detection was too broad and sometimes marked body lines as
# headings (e.g. "phủ phủ Lâm Bình...", "thành đến phía tả cung Khánh...").
# Because heading paragraphs are excluded from sentence export, those false headings
# created broken/non-terminal sentences and pages with final_lines but no final_sentences.

DNTC_V6_ADMIN_HEAD_RE = re.compile(
    r"^(?:HUYỆN|HUYEN|PHỦ|PHU|TỔNG|TONG|XÃ|XA|THÀNH|THANH|NÚI|NUI|SÔNG|SONG|"
    r"ĐỀN|DEN|CHÙA|CHUA|MIẾU|MIEU|LĂNG|LANG|CẦU|CAU|ĐÒ|DO|TỈNH|TINH)\b",
    re.I,
)
DNTC_V6_DOMAIN_HEADING_RE = re.compile(
    r"^(?:DỰNG ĐẶT|DUNG DAT|DIÊN CÁCH|DIEN CACH|PHONG TỤC|PHONG TUC|THÀNH TRÌ|THANH TRI|"
    r"NÚI SÔNG|NUI SONG|SÔNG NGÒI|SONG NGOI|ĐẦM AO|DAM AO|CẦU CỐNG|CAU CONG|"
    r"QUAN TẤN|QUAN TAN|ĐÊ ĐẬP|DE DAP|CHỢ QUÁN|CHO QUAN|TRƯỜNG HỌC|TRUONG HOC|"
    r"TỪ MIẾU|TU MIEU|CHÙA QUÁN|CHUA QUAN|LĂNG MỘ|LANG MO|CỔ TÍCH|CO TICH|"
    r"NHÂN VẬT|NHAN VAT|THỔ SẢN|THO SAN|HỘ KHẨU|HO KHAU|ĐIỀN THỔ|DIEN THO|"
    r"THUẾ LỆ|THUE LE|MỤC LỤC|MUC LUC|BÀI TỰ|BAI TU|LỜI NÓI ĐẦU|LOI NOI DAU)\b",
    re.I,
)
DNTC_V6_BODY_WORDS_IN_HEADING_RE = re.compile(
    r"\b(?:cách|giáp|thuộc|đời|năm|là|ở|có|cho|đến|từ|theo|gọi|đổi|lãnh|"
    r"phía|đông|tây|nam|bắc|huyện|phủ|xã|thôn|tổng|dặm)\b",
    re.I,
)

def _v6_upper_ratio(s: str) -> float:
    letters = LETTERS_RE.findall(norm_text(s))
    if not letters:
        return 0.0
    return sum(1 for ch in letters if ch.upper() == ch) / len(letters)

_ORIG_apply_dntc_reflow_corrections_v6 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v6(text)
    if not s:
        return s

    # Remove duplicate opener injected/preserved around Bai Tu pages.
    s = re.sub(r"(Trộm\s+nghĩ\s*:\s*[^.!?]{0,140}?\b(?:車書)?\s*)Trộm\s+nghĩ\s*(\(\s*1\s*\)\s*)?", r"\1\2", s, flags=re.I)
    s = re.sub(r"thiên\s*-?\s*Trộm\s+nghĩ\s+hạ", "thiên hạ", s, flags=re.I)

    # Old-scan administrative heading corrections.
    s = re.sub(r"\bLƯƠNG[.]SƠN\b", "LƯƠNG-SƠN", s, flags=re.I)
    s = re.sub(r"\bTHANH\s*-\s*CHU['’]?O?NG\b", "THANH-CHƯƠNG", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phi|phả|phd|phe)\s+ki[ée]m\s*-\s*l[yý]\)", "(do phủ kiêm-lý)", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phi|phả|phd|phe)\s+(?:th[eé]ng|th[oô]ng|hống)\s*-\s*(?:hgt|hạt)\)", "(do phủ thống-hạt)", s, flags=re.I)
    s = re.sub(r"\b(?:th[eé]ng|hống)\s*-\s*h(?:gt|ạt)\b", "thống-hạt", s, flags=re.I)
    s = re.sub(r"\bphd\b", "phủ", s, flags=re.I)
    s = re.sub(r"\bphi\s+(?=Nam\s+Linh|Tân\s+Bình|kiêm|thống)", "phủ ", s, flags=re.I)
    s = re.sub(r"\bph[ảa]\s+(?=kiêm|thống)", "phủ ", s, flags=re.I)

    # Old-scan numeric OCR corrections, context-limited.
    s = re.sub(r"\b(?:r8so|18so)\b", "1850", s, flags=re.I)
    s = re.sub(r"\br886\b", "1886", s, flags=re.I)
    s = re.sub(r"\br8og\b", "1899", s, flags=re.I)
    s = re.sub(r"\bthứ\s+1o\b", "thứ 10", s, flags=re.I)
    s = re.sub(r"\bthứ\s+ro\b", "thứ 10", s, flags=re.I)
    s = re.sub(r"\bthứ\s+r1\b", "thứ 11", s, flags=re.I)
    s = re.sub(r"\bthứ\s+rz\b", "thứ 12", s, flags=re.I)
    s = re.sub(r"\bthứ\s+r4\b", "thứ 14", s, flags=re.I)
    s = re.sub(r"\bthứ\s+r8\b", "thứ 18", s, flags=re.I)
    s = re.sub(r"\b([0-9]+)o(?=\s+(?:người|mẫu|quan|đồng|dặm|xã|thôn|tổng|hộc|thăng|hạp|thước))", r"\g<1>0", s, flags=re.I)
    s = re.sub(r"\bo4(?=\s+dặm)", "94", s, flags=re.I)
    s = re.sub(r"\b8o(?=\s+dặm)", "80", s, flags=re.I)
    s = re.sub(r"\b8s(?=\s+dặm)", "85", s, flags=re.I)
    s = re.sub(r"\bs8(?=\s+dặm)", "58", s, flags=re.I)
    s = re.sub(r"\bs2(?=\s+dặm)", "52", s, flags=re.I)
    s = re.sub(r"\bss(?=\s+dặm)", "55", s, flags=re.I)
    s = re.sub(r"\bso(?=\s+dặm)", "50", s, flags=re.I)
    s = re.sub(r"\b6;\s*xã\b", "67 xã", s, flags=re.I)
    s = re.sub(r"\b([0-9]+)\s*;\s*(?=xã|thôn)", r"\1", s, flags=re.I)
    s = re.sub(r"\b([0-9])\s*tồng\b", r"\1 tổng", s, flags=re.I)
    s = re.sub(r"\b([0-9])tổng\b", r"\1 tổng", s, flags=re.I)
    s = re.sub(r"\bs\s+(?:tồng|tổng)\b", "5 tổng", s, flags=re.I)
    s = re.sub(r"\bdim\b", "dặm", s, flags=re.I)
    s = re.sub(r"\bNaylnh\b", "Nay lĩnh", s, flags=re.I)
    s = re.sub(r"\blĩnh\s+s\s+tổng\b", "lĩnh 5 tổng", s, flags=re.I)
    s = re.sub(r"\bDến\b", "Đến", s)
    s = re.sub(r"\blệthuộc\b", "lệ thuộc", s, flags=re.I)
    s = re.sub(r"\blàhuyện\b", "là huyện", s, flags=re.I)
    s = re.sub(r"\bTrihuyện\b", "Tri huyện", s)
    s = re.sub(r"\bđồilà\b", "đổi là", s, flags=re.I)
    s = re.sub(r"\b(lại|mới|sau|rồi)\s+đồi\b", r"\1 đổi", s, flags=re.I)

    # Cleanup OCR debris around headings and sentence starts.
    s = re.sub(r"^[\s_—\-|lI1]+(?=HUYỆN|HUYEN|PHỦ|PHU|TỈNH|TINH|PHẦN|PHAN)", "", s, flags=re.I)
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([,;:!?])(?=\S)", r"\1 ", s)
    s = re.sub(r"(?<!\d)\.(?=\S)", ". ", s)
    s = re.sub(r"\s*-\s*", "-", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

_ORIG_is_heading_text_v6 = is_heading_text

def is_heading_text(s: str) -> bool:
    s0 = apply_dntc_reflow_corrections(norm_text(s)).strip()
    core = s0.strip(" |_\u2014\u2013-")
    if not core or len(core) > 130:
        return False

    # Critical fix: never promote lowercase sentence-like lines to headings.
    # Use Unicode-aware islower(); regex ranges such as à-ỹ also include some uppercase codepoints.
    first_alpha = next((ch for ch in core if ch.isalpha()), "")
    if first_alpha and first_alpha.islower():
        return False

    upper_ratio = _v6_upper_ratio(core)

    # Lines with normal sentence punctuation are not headings unless they are almost all uppercase.
    if re.search(r"[,.!?;]", core) and upper_ratio < 0.85:
        return False

    # Canonical all-caps/admin headings. Parenthetical "(do phủ ...)" is allowed.
    if DNTC_V6_ADMIN_HEAD_RE.match(core):
        head_part = core.split("(")[0].strip()
        if len(core) <= 120 and (upper_ratio >= 0.55 or re.match(r"^(HUYỆN|HUYEN|PHỦ|PHU)\s+[A-ZÀ-ỸĐ0-9 .'’-]+", head_part)):
            # Reject body-looking lines that start with an admin word but continue as prose.
            if not DNTC_V6_BODY_WORDS_IN_HEADING_RE.search(head_part.replace("HUYỆN", "").replace("PHỦ", "")):
                return True

    if DNTC_V6_DOMAIN_HEADING_RE.match(core) and len(core) <= 110 and upper_ratio >= 0.55:
        return True

    # General old heading: short and mostly uppercase.
    if len(core) <= 85 and upper_ratio >= 0.82 and not LIBRARY_NOISE_RE.search(core):
        return True

    return False

_ORIG_filter_line_v6 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    kept, ok, reasons = _ORIG_filter_line_v6(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if not ok or kept is None:
        return kept, ok, reasons
    fixed = apply_dntc_reflow_corrections(kept.get("text", ""))
    if not fixed:
        return None, False, (reasons or []) + ["empty_after_v6_correction"]
    kept["text"] = fixed
    # Critical fix: reclassify both ways, not only body -> heading.
    kept["line_type"] = "heading" if is_heading_text(fixed) else "body"
    q = line_quality(fixed, kept.get("ocr_conf"))
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    return kept, True, reasons

_ORIG_split_sentences_vietnamese_v6 = split_sentences_vietnamese

def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    if paragraph_type == "heading":
        return []
    out = _ORIG_split_sentences_vietnamese_v6(paragraph_text, paragraph_type)
    final = []
    for s in out:
        s = apply_dntc_reflow_corrections(s)
        if not s:
            continue
        q = text_metrics(s)
        # final_sentences_only must be strict. Non-terminal fragments remain in paragraphs/text,
        # but should not pollute the sentence-only CSV.
        if globals().get("STRICT_SENTENCE_ONLY", True):
            if not ends_strong_sentenceish(s):
                continue
            if looks_like_sentence_fragment(s) and q["word_count"] < 18:
                continue
        final.append(s)
    return final

print("DNTC v6 patch loaded: strict heading classifier, stricter sentence-only validation, paragraph text export recommended.")


DNTC v6 patch loaded: strict heading classifier, stricter sentence-only validation, paragraph text export recommended.


In [ ]:
# ============================================================
# 8F. DNTC v7 patch: backmatter + heading/number cleanup (no Han preservation)
# ============================================================
# This keeps the useful v7 fixes for Vietnamese extraction:
#   1) Publisher/backmatter catalog pages/lines are dropped more aggressively.
#   2) Old-scan administrative headings are detected more accurately.
#   3) Old-scan numeric/OCR artifacts are normalized in Vietnamese text.
#
# Han text-layer preservation is intentionally NOT enabled in this variant.
# Pages/lines are still judged by the normal Vietnamese-oriented pipeline.

DNTC_V7_BACKMATTER_RE = re.compile(
    r"(?:V\.?\s*H\.?\s*T\.?\s*T\.?|Vietnam\s+Culture\s+Series|HIGHER\s+EDUCATION\s+IN\s+THE\s+REPUBLIC|"
    r"LA\s+LITTERATURE\s+VIETNAMIENNE|INTRODUCTION\s+TO\s+VIETNAMESE|"
    r"do\s+Nha\s+V[ăa]n\s*-\s*H[oó]a.*xu[aấ]t\s+b[aả]n|"
    r"c[oó]\s+b[aá]n\s+t[aạ]i\s+c[aá]c\s+n[oơ]i|"
    r"Nh[ữu]ng\s+t[aậ]p\s+V[ĂA]N\s*H[ÓO]A|"
    r"M[ỤU]C\s*L[ỤU]C|MUC\s*LUC|S[ốo]\s+đ[ăa]ng\s+k[ií]\s+KHXB|"
    r"Quy[ếe]t\s+đ[iị]nh\s+xu[aấ]t\s+b[aả]n|In\s+\d+\s+cu[ốo]n)",
    re.I,
)

DNTC_V7_ADMIN_HEADING_RE = re.compile(
    r"^\s*[_\"'“”‘’\-\u2013\u2014.]*\s*"
    r"(?:HUYỆN|HUYEN|huyện\s+[A-ZÀ-ỸĐ]|PHỦ|PHU|TỔNG|TONG|TỈNH|TINH|"
    r"NÚI|NUI|SÔNG|SONG|GIANG|KHE|ĐẦM|DAM|CẢNG|CANG|CẦU|CAU|ĐÒ|DO|"
    r"ĐỀN|DEN|CHÙA|CHUA|MIẾU|MIEU|LĂNG|LANG|ĐÀN|DAN|THÀNH|THANH|"
    r"VĂN\s*-?\s*MIẾU|VAN\s*-?\s*MIEU|PHỤ\s+LỤC|PHU\s+LUC)\b",
    re.I,
)

DNTC_V7_STRUCTURAL_START_RE = re.compile(
    r"\b(?:HUYỆN|HUYEN|PHỦ|PHU|TỔNG|TONG|TỈNH|TINH|NÚI|NUI|SÔNG|SONG|"
    r"CẨN\s+ÁN|CAN\s+AN|XÉT|XET|NĂM|NAM|ĐỜI|DOI|Ở|O|PHÍA|PHIA|"
    r"TỪ|TU|LẠI|LAI|ĐẾN|DEN|TỪNG|TUNG|ĐÀN|DAN)\b",
    re.I,
)

_ORIG_is_backmatter_page_v7 = is_backmatter_page

def is_backmatter_page(text: str, text_m: dict) -> bool:
    s = norm_text(text)
    if not s:
        return False
    # Catalog / publisher ad pages are not book content even when OCR looks high quality.
    if DNTC_V7_BACKMATTER_RE.search(s):
        return True
    return _ORIG_is_backmatter_page_v7(text, text_m)


_ORIG_apply_dntc_reflow_corrections_v7 = apply_dntc_reflow_corrections

def apply_dntc_reflow_corrections(text: str) -> str:
    s = _ORIG_apply_dntc_reflow_corrections_v7(text)
    if not s:
        return s

    # Remove remaining scanning/catalog symbols.
    s = re.sub(r"(^|\s)\([RC]\)(?=\s|$)", " ", s)
    s = re.sub(r"[|_]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # Normalize headings and admin parentheticals from old scans.
    s = re.sub(r"^\s*huyện\s+(?=[A-ZÀ-ỸĐ])", "HUYỆN ", s)
    s = re.sub(r"^\s*phủ\s+(?=[A-ZÀ-ỸĐ])", "PHỦ ", s)
    s = re.sub(r"^\s*tổng\s+(?=[A-ZÀ-ỸĐ])", "TỔNG ", s)
    s = re.sub(r"\bTHANH\s*-\s*CHU(?:O['’]?|Ơ)NG\b", "THANH-CHƯƠNG", s, flags=re.I)
    s = re.sub(r"\bLAN\s*NAM\b", "La-Nam", s, flags=re.I)
    s = re.sub(r"\bThồ\s*[.]\s*Du\b", "Thổ-Du", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phi|phả|phá|phd|phe)\s*[: ]+\s*(?:th[eé]ng|th[oô]ng|hống)\s*-\s*(?:hgt|hạt)\)", "(do phủ thống-hạt)", s, flags=re.I)
    s = re.sub(r"\((?:do|đo)\s+(?:phi|phả|phá|phd|phe)\s+(?:ki[ée]m)\s*-\s*l[yý]\)", "(do phủ kiêm-lý)", s, flags=re.I)
    s = re.sub(r"\b(?:phi|phả|phá|phd|phe)\s+ki[ée]m\s*-\s*l[yý]\b", "phủ kiêm-lý", s, flags=re.I)
    s = re.sub(r"\b(?:phi|phả|phá|phd|phe)\s+(?:th[eé]ng|th[oô]ng|hống)\s*-\s*(?:hgt|hạt)\b", "phủ thống-hạt", s, flags=re.I)

    # Numeric OCR cleanup. Apply only where the surrounding context is administrative/statistical.
    s = re.sub(r"(?<=\d)[.]\s+(?=\d)", ".", s)
    s = re.sub(r"\bhơn\s+6s\b", "hơn 65", s, flags=re.I)
    s = re.sub(r"\bcó\s*8o\b", "có 80", s, flags=re.I)
    s = re.sub(r"\bcộng\s+4o\b", "cộng 40", s, flags=re.I)
    s = re.sub(r"\bcó\s+4s\b", "có 45", s, flags=re.I)
    s = re.sub(r"\b1oo(?=[.\s])", "100", s, flags=re.I)
    s = re.sub(r"\b([0-9]+)o(?=[.\s,]*(?:người|mẫu|quan|đồng|dặm|xã|thôn|tổng|hộc|thăng|hạp|thước|tiền|đ))", r"\g<1>0", s, flags=re.I)
    s = re.sub(r"\b([0-9]+)s(?=[.\s,]*(?:người|mẫu|quan|đồng|dặm|xã|thôn|tổng|hộc|thăng|hạp|thước|tiền|đ))", r"\g<1>5", s, flags=re.I)
    s = re.sub(r"\bthứ\s*[¡!|i]r\b", "thứ 11", s, flags=re.I)
    s = re.sub(r"\br1\b", "11", s, flags=re.I)
    s = re.sub(r"\br8\b", "18", s, flags=re.I)
    s = re.sub(r"\brạo6\b", "1906", s, flags=re.I)
    s = re.sub(r"\bTién\b", "Tiền", s, flags=re.I)
    s = re.sub(r"\bthu[€é]\b", "thuế", s, flags=re.I)
    s = re.sub(r"\bthuổ\b", "thuế", s, flags=re.I)
    s = re.sub(r"\bđiền\s+thd\b", "điền thổ", s, flags=re.I)
    s = re.sub(r"\bdin\s+thd\b", "điền thổ", s, flags=re.I)
    s = re.sub(r"\bm4u\b", "mẫu", s, flags=re.I)
    s = re.sub(r"\b(?:céng|cong)\b", "cộng", s, flags=re.I)
    s = re.sub(r"\bhyp\s+cộng\b", "hợp cộng", s, flags=re.I)
    s = re.sub(r"\bl\s+đã\b", "đã", s, flags=re.I)
    s = re.sub(r"\bHồi\s+[—–-]+\s*thuộc\b", "Hồi thuộc", s, flags=re.I)
    s = re.sub(r"\s+([,.;:!?])", r"\1", s)
    s = re.sub(r"([.!?])\s*[.]\s*", r"\1 ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


_ORIG_is_heading_text_v7 = is_heading_text

def is_heading_text(s: str) -> bool:
    s0 = apply_dntc_reflow_corrections(norm_text(s)).strip()
    core = s0.strip(" |_\u2014\u2013- .")
    if not core or len(core) > 140:
        return False

    if _ORIG_is_heading_text_v7(core):
        return True

    upper_ratio = _v6_upper_ratio(core) if "_v6_upper_ratio" in globals() else 0.0
    head_part = core.split("(")[0].strip(" -–—:.")

    # Accept OCR-old administrative headings, including lowercase "huyện" followed by all-caps toponym.
    if DNTC_V7_ADMIN_HEADING_RE.match(core) and len(core) <= 130:
        has_terminal_sentence = bool(re.search(r"[.!?]\s*$", core))
        # parenthetical "(do phủ ...)" is common in true huyện headings.
        has_admin_parenthesis = bool(re.search(r"\(\s*(?:do|đo)\s+phủ\s+(?:kiêm-lý|thống-hạt)\s*\)", core, re.I))
        starts_lower_admin_heading = bool(re.match(r"^\s*huyện\s+[A-ZÀ-ỸĐ]", core))
        if not has_terminal_sentence and (upper_ratio >= 0.48 or has_admin_parenthesis or starts_lower_admin_heading):
            # Reject obvious prose lines like "Huyện đặt ở..." / "Phủ doãn..."
            if not re.match(r"^\s*(?:Huyện|Phủ|Tổng)\s+(?:đặt|này|ấy|doãn|thuộc|là|có|gồm)\b", core, re.I):
                return True

    # Short all-caps toponym labels with explanatory parenthesis are headings/entry labels.
    if len(core) <= 95 and upper_ratio >= 0.60 and re.search(r"\([^)]{2,50}\)", core):
        if not re.search(r"\b(?:cách|giáp|đời|năm|thuộc|đến|từ)\b", head_part, re.I):
            return True

    return False


_ORIG_filter_line_v7 = filter_line

def filter_line(line: dict, page_class: str, correction_log: list, doc=None, reocr_state=None) -> tuple:
    raw = norm_text(line.get("raw_text") or line.get("text") or "")
    if not raw:
        return None, False, ["empty"]

    # Hard-drop publisher catalog lines that survive page classification.
    if DNTC_V7_BACKMATTER_RE.search(raw):
        return None, False, ["publisher_backmatter_line_v7"]

    kept, ok, reasons = _ORIG_filter_line_v7(line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
    if not ok or kept is None:
        return kept, ok, reasons

    fixed = apply_dntc_reflow_corrections(kept.get("text", ""))
    if not fixed or DNTC_V7_BACKMATTER_RE.search(fixed):
        return None, False, (reasons or []) + ["empty_or_backmatter_after_v7_correction"]
    kept["text"] = fixed
    kept["line_type"] = "heading" if is_heading_text(fixed) else "body"
    q = text_metrics(fixed)
    kept["line_quality_score"] = q["quality_score"]
    kept["line_vietnamese_ratio"] = q["vietnamese_ratio"]
    kept["line_weird_char_ratio"] = q["weird_char_ratio"]
    return kept, True, reasons


_ORIG_split_sentences_vietnamese_v7 = split_sentences_vietnamese

def _v7_split_overlong_sentence(s: str) -> list:
    s = apply_dntc_reflow_corrections(s)
    if len(s) <= 850:
        return [s]
    # Split only on strong punctuation followed by obvious structural starts.
    parts = re.split(
        r"(?<=[.!?])\s+(?=(?:Cẩn\s+án|Xét|Năm\s+[A-ZÀ-ỸĐa-zà-ỹđ]|Đời\s+[A-ZÀ-ỸĐa-zà-ỹđ]|"
        r"Phía\s+|Ở\s+|Từng\s+|Đàn\s+|Huyện\s+|Phủ\s+|Núi\s+|Sông\s+))",
        s,
        flags=re.I,
    )
    if len(parts) == 1:
        return [s]
    out = []
    buf = ""
    for p in parts:
        p = p.strip()
        if not p:
            continue
        if not buf:
            buf = p
        elif len(buf) < 80:
            buf = buf + " " + p
        else:
            out.append(buf)
            buf = p
    if buf:
        out.append(buf)
    return out


def split_sentences_vietnamese(paragraph_text: str, paragraph_type="body") -> list:
    if paragraph_type == "heading":
        return []
    base = _ORIG_split_sentences_vietnamese_v7(paragraph_text, paragraph_type)
    final = []
    for s in base:
        for part in _v7_split_overlong_sentence(s):
            part = apply_dntc_reflow_corrections(part)
            if not part:
                continue
            if globals().get("STRICT_SENTENCE_ONLY", True):
                if not ends_strong_sentenceish(part):
                    continue
                q = text_metrics(part)
                if looks_like_sentence_fragment(part) and q["word_count"] < 18:
                    continue
            final.append(part)
    return final

print("DNTC v7 patch loaded: backmatter hardened, admin headings/numeric cleanup improved; Han text-layer preservation disabled.")


In [14]:

# ============================================================
# 9. Process one PDF and all PDFs
# ============================================================
all_final_lines = []
all_paragraphs = []
all_sentences = []
page_quality_report = []
source_selection_report = []
line_filter_audit = []
dropped_pages = []
dropped_lines = []
correction_log = []
suspicious_lines = []
suspicious_sentences = []


def make_work_id(pdf_path: Path, used: set) -> str:
    base = re.sub(r"[^A-Za-z0-9_\-]+", "_", pdf_path.stem).strip("_") or "pdf"
    wid = base
    k = 2
    while wid in used:
        wid = f"{base}_{k}"
        k += 1
    used.add(wid)
    return wid


def suspicious_reasons_for_text(text: str, q: dict) -> list:
    reasons = []
    if CLEAR_JUNK_RE.search(norm_text(text)):
        reasons.append("known_junk_pattern")
    if q["quality_score"] < MIN_KEEP_SENTENCE_QUALITY:
        reasons.append(f"low_quality:{q['quality_score']:.1f}")
    if q["weird_char_ratio"] > 0.035:
        reasons.append("weird_char_ratio")
    if q["word_count"] >= 8 and q["vietnamese_ratio"] < MIN_VIET_RATIO_FOR_LONG_TEXT:
        reasons.append("low_vietnamese_ratio")
    if q["repeated_char_ngram_ratio"] > 0.05:
        reasons.append("repeated_ngram_ratio")
    if q["library_noise_score"] >= MAX_LIBRARY_NOISE_SCORE:
        reasons.append("library_noise")
    if q["unaccented_vi_hits"] >= 3 and q["accent_ratio"] < 0.015:
        reasons.append("possible_missing_diacritics")
    if globals().get("STRICT_SENTENCE_ONLY", True):
        try:
            if looks_like_sentence_fragment(text):
                reasons.append("sentence_fragment_shape")
            if not ends_hard_sentence(text) and q["word_count"] < 18:
                reasons.append("no_terminal_punctuation")
        except Exception:
            pass
    return reasons


def process_pdf(pdf_path: Path, work_id: str, remaining_page_budget=None):
    print(f"\nProcessing {work_id}: {pdf_path}")
    try:
        doc = fitz.open(str(pdf_path))
    except Exception as e:
        dropped_pages.append({"work_id": work_id, "pdf_path": str(pdf_path), "page_number": None, "reason": f"cannot_open:{type(e).__name__}"})
        print("Cannot open PDF:", e)
        return 0

    total_pages = len(doc)
    content_start_page = estimate_content_start(doc, work_id, pdf_path)
    print("pages:", total_pages, "auto_content_start_page:", content_start_page)
    page_limit = total_pages if MAX_PAGES_PER_PDF is None else min(total_pages, MAX_PAGES_PER_PDF)
    if remaining_page_budget is not None:
        page_limit = min(page_limit, remaining_page_budget)
    reocr_state = {"used": 0}
    processed_pages = 0

    for page_idx in tqdm(range(page_limit), desc=work_id):
        processed_pages += 1
        page = doc[page_idx]
        page_number = page_idx + 1
        visual_m = visual_page_metrics(page)
        tl_lines_for_class = extract_pdf_text_layer_lines(page, page_idx, work_id, pdf_path)
        tl_text_for_class = lines_to_text(tl_lines_for_class)
        tl_m_for_class = text_metrics(tl_text_for_class, lines=[r.get("raw_text", "") for r in tl_lines_for_class])
        page_class, class_reasons = classify_page(page_number, tl_text_for_class, tl_m_for_class, visual_m, content_start_page)

        # If text layer is too poor to classify but visual has nonblank text-like page, do a quick OCR for classification.
        classification_text = tl_text_for_class
        classification_m = tl_m_for_class
        if page_class == "junk_ocr_page" and tesseract_available() and not is_blank_page(tl_m_for_class, visual_m):
            quick_lines = extract_tesseract_lines(page, page_idx, work_id, pdf_path, dpi=FAST_FRONTMATTER_OCR_DPI)
            quick_text = lines_to_text(quick_lines)
            quick_m = text_metrics(quick_text, lines=[r.get("raw_text", "") for r in quick_lines], ocr_conf_values=[r.get("ocr_conf") for r in quick_lines])
            if quick_m["quality_score"] > classification_m["quality_score"]:
                classification_text, classification_m = quick_text, quick_m
                page_class, class_reasons = classify_page(page_number, classification_text, classification_m, visual_m, content_start_page)

        report = {
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
            "content_start_page": content_start_page, "page_class": page_class,
            "class_reasons": ";".join(class_reasons) if class_reasons else "",
            **{f"visual_{k}": v for k, v in visual_m.items()},
            **{f"class_text_{k}": v for k, v in classification_m.items()},
        }

        if page_class in {"blank_page", "patterned_endpaper", "library_barcode_stamp_watermark", "cover_title_front_matter", "front_matter_before_content", "publisher_backmatter_page"}:
            report.update({"selected_source": "dropped_by_page_classifier", "kept_lines": 0, "dropped_lines": 0, "final_sentences": 0})
            page_quality_report.append(report)
            dropped_pages.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "page_class": page_class, "drop_reason": ";".join(class_reasons) if class_reasons else page_class,
                **classification_m,
            })
            continue

        selected_lines, selected_source, selected_m, tl_m, ocr_m, source_reason = select_page_source(page, page_idx, work_id, pdf_path, page_class)
        source_selection_report.append({
            "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
            "page_class": page_class, "selected_source": selected_source, "source_reason": source_reason,
            **{f"text_layer_{k}": v for k, v in tl_m.items()},
            **{f"ocr_{k}": v for k, v in (ocr_m or {}).items()},
        })

        if not selected_lines or (DROP_LOW_CONF_OCR_PAGES and selected_m.get("quality_score", 0) < MIN_SELECTED_PAGE_QUALITY):
            report.update({"selected_source": selected_source, "kept_lines": 0, "dropped_lines": len(selected_lines), "final_sentences": 0})
            page_quality_report.append(report)
            dropped_pages.append({
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "page_class": "junk_ocr_page" if selected_source == "drop_no_reliable_source" else page_class,
                "drop_reason": f"low_selected_source_quality:{selected_m.get('quality_score', 0):.1f};{source_reason}",
                **selected_m,
            })
            continue

        kept_lines = []
        dropped_count = 0
        seen_line_texts = Counter()
        for li, raw_line in enumerate(selected_lines):
            raw_line = dict(raw_line)
            raw_line["line_global_id"] = f"{work_id}_p{page_number:04d}_l{li+1:03d}"
            kept, is_kept, reasons = filter_line(raw_line, page_class, correction_log, doc=doc, reocr_state=reocr_state)
            audit_row = {
                "work_id": work_id, "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                "line_id": raw_line["line_global_id"], "source": raw_line.get("source"), "page_class": page_class,
                "raw_text": raw_line.get("raw_text", ""), "kept": bool(is_kept), "reasons": ";".join(reasons),
                "bbox": json.dumps(raw_line.get("bbox", []), ensure_ascii=False), "ocr_conf": raw_line.get("ocr_conf"),
            }
            if is_kept and kept is not None:
                # Drop duplicate exact line repeated on same page, except headings.
                key = normalize_for_match(kept["text"])
                seen_line_texts[key] += 1
                if seen_line_texts[key] > 2 and kept.get("line_type") != "heading":
                    is_kept = False
                    reasons = reasons + ["duplicate_repeated_line"]
                    audit_row.update({"kept": False, "reasons": ";".join(reasons), "final_text": kept["text"]})
                else:
                    kept["page_class"] = page_class
                    kept["selected_source"] = selected_source
                    kept["source_reason"] = source_reason
                    kept_lines.append(kept)
                    audit_row.update({"final_text": kept["text"], "line_quality_score": kept.get("line_quality_score")})
                    sr = suspicious_reasons_for_text(kept["text"], line_quality(kept["text"], kept.get("ocr_conf")))
                    if sr:
                        suspicious_lines.append({**audit_row, "suspicious_reasons": ";".join(sr)})
            if not is_kept:
                dropped_count += 1
                dropped_lines.append({**audit_row, "kept": False, "reasons": ";".join(reasons)})
            line_filter_audit.append(audit_row)

        paragraphs = reflow_lines_to_paragraphs(kept_lines, page.rect)
        page_sentence_count = 0
        for kline in kept_lines:
            row = dict(kline)
            row["bbox"] = json.dumps(row.get("bbox", []), ensure_ascii=False)
            all_final_lines.append(row)

        for pi, para in enumerate(paragraphs):
            paragraph_id = f"{work_id}_p{page_number:04d}_para{pi+1:03d}"
            prow = {
                "paragraph_id": paragraph_id, "work_id": work_id, "pdf_path": str(pdf_path),
                "page_idx": page_idx, "page_number": page_number, **para,
                "bbox": json.dumps(para.get("bbox", []), ensure_ascii=False),
            }
            all_paragraphs.append(prow)
            if para["paragraph_type"] == "heading":
                continue
            sentences = split_sentences_vietnamese(para["text"], para["paragraph_type"])
            for si, sent in enumerate(sentences):
                sq = text_metrics(sent)
                s_reasons = suspicious_reasons_for_text(sent, sq)
                sent_id = f"{work_id}_s{len(all_sentences)+1:07d}"
                srow = {
                    "sent_id": sent_id, "paragraph_id": paragraph_id, "work_id": work_id,
                    "pdf_path": str(pdf_path), "page_idx": page_idx, "page_number": page_number,
                    "sentence_index_in_paragraph": si, "text": sent, "source": para.get("source"),
                    "paragraph_type": para.get("paragraph_type"), "bbox": prow["bbox"],
                    "quality_score": sq["quality_score"], "vietnamese_ratio": sq["vietnamese_ratio"],
                    "weird_char_ratio": sq["weird_char_ratio"], "dictionary_hit_ratio": sq["dictionary_hit_ratio"],
                    "suspicious_reasons": ";".join(s_reasons),
                }
                if s_reasons:
                    suspicious_sentences.append(srow)
                    # Keep suspicious if it is still above hard floor; otherwise drop from final sentence-only.
                    if (sq["quality_score"] < MIN_KEEP_SENTENCE_QUALITY or
                        "sentence_fragment_shape" in s_reasons or "no_terminal_punctuation" in s_reasons):
                        continue
                all_sentences.append(srow)
                page_sentence_count += 1

        report.update({
            "selected_source": selected_source, "source_reason": source_reason,
            **{f"selected_{k}": v for k, v in selected_m.items()},
            "kept_lines": len(kept_lines), "dropped_lines": dropped_count,
            "paragraphs": len(paragraphs), "final_sentences": page_sentence_count,
            "line_reocr_used_pdf": reocr_state.get("used", 0),
        })
        page_quality_report.append(report)

    doc.close()
    return processed_pages

used_ids = set()
page_budget = FAST_TEST_MAX_PAGES_TOTAL if FAST_TEST_MODE else None
start_time = time.time()
processed_pages_total = 0

for pdf_i, pdf_path in enumerate(pdf_paths, 1):
    if page_budget is not None and page_budget <= 0:
        break
    work_id = make_work_id(pdf_path, used_ids)
    n = process_pdf(pdf_path, work_id, remaining_page_budget=page_budget)
    processed_pages_total += n
    if page_budget is not None:
        page_budget -= n

elapsed = time.time() - start_time
print("\nDone processing")
print("processed_pages:", processed_pages_total)
print("final_lines:", len(all_final_lines))
print("paragraphs:", len(all_paragraphs))
print("sentences:", len(all_sentences))
print("dropped_pages:", len(dropped_pages))
print("dropped_lines:", len(dropped_lines))
print("elapsed_min:", round(elapsed / 60, 2))




Processing 01: /kaggle/working/dntc_auto/raw_drive/01.pdf
pages: 136 auto_content_start_page: 7


01:   0%|          | 0/136 [00:00<?, ?it/s]


Processing 05: /kaggle/working/dntc_auto/raw_drive/05.pdf
pages: 141 auto_content_start_page: 27


05:   0%|          | 0/141 [00:00<?, ?it/s]


Processing 07_08: /kaggle/working/dntc_auto/raw_drive/07_08.pdf
pages: 220 auto_content_start_page: 9


...


Processing q2_3_4: /kaggle/working/dntc_auto/raw_drive/q2_3_4.pdf
pages: 502 auto_content_start_page: 3


q2_3_4:   0%|          | 0/502 [00:00<?, ?it/s]


Processing q6: /kaggle/working/dntc_auto/raw_drive/q6.pdf
pages: 528 auto_content_start_page: 3


q6:   0%|          | 0/528 [00:00<?, ?it/s]


Done processing
processed_pages: 2643
final_lines: 57835
paragraphs: 10513
sentences: 17197
dropped_pages: 255
dropped_lines: 15323
elapsed_min: 118.16


In [15]:

# ============================================================
# 9B. DNTC v5 post-process: stitch cross-page paragraphs and rebuild sentences
# ============================================================
# Processing is page-local for speed, but old books frequently continue a sentence
# from the bottom of one page to the top of the next. This pass stitches adjacent
# body paragraphs before exporting final_paragraphs/final_sentences_only.

CROSS_PAGE_DANGLING_END_RE = re.compile(
    r"\b(?:làm\s+chỗ|đề|để|mỗi|của|và|ở|là|theo|có|gồm|đem|cho|về|trong|ngoài|phía|cách)\s*$",
    re.I,
)
CROSS_PAGE_LOWER_START_RE = re.compile(r"^[a-zà-ỹđ]", re.I)


def should_stitch_paragraph_rows(prev: dict, cur: dict) -> bool:
    if not prev or not cur:
        return False
    if str(prev.get("work_id")) != str(cur.get("work_id")):
        return False
    if prev.get("paragraph_type") == "heading" or cur.get("paragraph_type") == "heading":
        return False
    pt = apply_dntc_reflow_corrections(str(prev.get("text", "")))
    ct = apply_dntc_reflow_corrections(str(cur.get("text", "")))
    if not pt or not ct:
        return False
    # Only adjacent flow; across page or same page after geometry produced false split.
    try:
        pp = int(prev.get("page_number"))
        cp = int(cur.get("page_number"))
        if cp < pp or cp > pp + 1:
            return False
    except Exception:
        pass
    if CROSS_PAGE_DANGLING_END_RE.search(pt):
        return True
    if (not ends_hard_sentence(pt)) and (CROSS_PAGE_LOWER_START_RE.match(ct) or begins_continuation(ct)):
        return True
    return False


def stitch_paragraph_rows(rows: list) -> list:
    if not rows:
        return rows
    rows = sorted(rows, key=lambda r: (str(r.get("work_id", "")), int(r.get("page_number", 0) or 0), int(r.get("paragraph_index_in_page", 0) or 0)))
    stitched = []
    for r in rows:
        r = dict(r)
        r["text"] = apply_dntc_reflow_corrections(str(r.get("text", "")))
        if stitched and should_stitch_paragraph_rows(stitched[-1], r):
            prev = stitched[-1]
            prev["text"] = apply_dntc_reflow_corrections(str(prev.get("text", "")) + " " + str(r.get("text", "")))
            prev["line_count"] = int(prev.get("line_count", 0) or 0) + int(r.get("line_count", 0) or 0)
            prev["line_ids"] = ";".join(x for x in [str(prev.get("line_ids", "")), str(r.get("line_ids", ""))] if x and x != "nan")
            prev["stitched_from"] = ";".join(x for x in [str(prev.get("stitched_from", "")), str(r.get("paragraph_id", ""))] if x and x != "nan")
            prev["page_span"] = f"{prev.get('page_number')}-{r.get('page_number')}" if prev.get("page_number") != r.get("page_number") else str(prev.get("page_number"))
        else:
            r.setdefault("stitched_from", "")
            r.setdefault("page_span", str(r.get("page_number", "")))
            stitched.append(r)
    return stitched


def rebuild_sentences_from_paragraphs():
    global all_paragraphs, all_sentences, suspicious_sentences
    old_para_count = len(all_paragraphs)
    old_sent_count = len(all_sentences)
    all_paragraphs = stitch_paragraph_rows(all_paragraphs)
    all_sentences = []
    suspicious_sentences = []
    sent_counter = Counter()
    for para in all_paragraphs:
        if para.get("paragraph_type") == "heading":
            continue
        sentences = split_sentences_vietnamese(para.get("text", ""), para.get("paragraph_type", "body"))
        for si, sent in enumerate(sentences):
            sent = apply_dntc_reflow_corrections(sent)
            sq = text_metrics(sent)
            s_reasons = suspicious_reasons_for_text(sent, sq)
            # Hard fragment blocking remains in final, but keep audit rows.
            sent_counter[str(para.get("work_id"))] += 1
            sent_id = f"{para.get('work_id')}_s{sent_counter[str(para.get('work_id'))]:07d}"
            srow = {
                "sent_id": sent_id,
                "paragraph_id": para.get("paragraph_id"),
                "work_id": para.get("work_id"),
                "pdf_path": para.get("pdf_path"),
                "page_idx": para.get("page_idx"),
                "page_number": para.get("page_number"),
                "page_span": para.get("page_span", para.get("page_number")),
                "sentence_index_in_paragraph": si,
                "text": sent,
                "source": para.get("source"),
                "paragraph_type": para.get("paragraph_type"),
                "bbox": para.get("bbox"),
                "quality_score": sq["quality_score"],
                "vietnamese_ratio": sq["vietnamese_ratio"],
                "weird_char_ratio": sq["weird_char_ratio"],
                "dictionary_hit_ratio": sq["dictionary_hit_ratio"],
                "suspicious_reasons": ";".join(s_reasons),
            }
            if s_reasons:
                suspicious_sentences.append(srow)
                if (sq["quality_score"] < MIN_KEEP_SENTENCE_QUALITY or
                    "sentence_fragment_shape" in s_reasons or "no_terminal_punctuation" in s_reasons):
                    continue
            all_sentences.append(srow)
    print(f"DNTC v5 post-process: paragraphs {old_para_count} -> {len(all_paragraphs)}, sentences {old_sent_count} -> {len(all_sentences)}")

rebuild_sentences_from_paragraphs()


DNTC v5 post-process: paragraphs 10513 -> 9565, sentences 17197 -> 17630


In [16]:

# ============================================================
# 10. Export final files + audit + zip
# ============================================================
FINAL_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)
AUDIT_DIR.mkdir(parents=True, exist_ok=True)
PKG_DIR.mkdir(parents=True, exist_ok=True)

lines_df = pd.DataFrame(all_final_lines)
paras_df = pd.DataFrame(all_paragraphs)
sents_df = pd.DataFrame(all_sentences)
page_df = pd.DataFrame(page_quality_report)
source_df = pd.DataFrame(source_selection_report)
line_audit_df = pd.DataFrame(line_filter_audit)
dropped_pages_df = pd.DataFrame(dropped_pages)
dropped_lines_df = pd.DataFrame(dropped_lines)
correction_df = pd.DataFrame(correction_log)
susp_lines_df = pd.DataFrame(suspicious_lines)
susp_sents_df = pd.DataFrame(suspicious_sentences)

# Stable sort.
for df, cols in [
    (lines_df, ["work_id", "page_number", "line_global_id"]),
    (paras_df, ["work_id", "page_number", "paragraph_id"]),
    (sents_df, ["work_id", "page_number", "sent_id"]),
]:
    if not df.empty:
        use_cols = [c for c in cols if c in df.columns]
        df.sort_values(use_cols, inplace=True)
        df.reset_index(drop=True, inplace=True)

final_lines_csv = FINAL_DIR / "final_lines.csv"
final_paras_csv = FINAL_DIR / "final_paragraphs.csv"
final_sents_csv = FINAL_DIR / "final_sentences_only.csv"
final_sents_auto_csv = FINAL_DIR / "final_sentences_auto.csv"  # compatibility copy; sentence-only in this notebook.

page_report_csv = AUDIT_DIR / "page_quality_report.csv"
line_filter_csv = AUDIT_DIR / "line_filter_audit.csv"
correction_csv = AUDIT_DIR / "correction_log.csv"
susp_lines_csv = AUDIT_DIR / "suspicious_lines.csv"
susp_sents_csv = AUDIT_DIR / "suspicious_sentences.csv"
dropped_pages_csv = AUDIT_DIR / "dropped_pages.csv"
dropped_lines_csv = AUDIT_DIR / "dropped_lines.csv"
source_selection_csv = AUDIT_DIR / "source_selection_report.csv"
summary_by_work_csv = AUDIT_DIR / "summary_quality_by_work_id.csv"
summary_csv = AUDIT_DIR / "run_summary.csv"

lines_df.to_csv(final_lines_csv, index=False, encoding="utf-8-sig")
paras_df.to_csv(final_paras_csv, index=False, encoding="utf-8-sig")
sents_df.to_csv(final_sents_csv, index=False, encoding="utf-8-sig")
sents_df.to_csv(final_sents_auto_csv, index=False, encoding="utf-8-sig")
page_df.to_csv(page_report_csv, index=False, encoding="utf-8-sig")
line_audit_df.to_csv(line_filter_csv, index=False, encoding="utf-8-sig")
correction_df.to_csv(correction_csv, index=False, encoding="utf-8-sig")
susp_lines_df.to_csv(susp_lines_csv, index=False, encoding="utf-8-sig")
susp_sents_df.to_csv(susp_sents_csv, index=False, encoding="utf-8-sig")
dropped_pages_df.to_csv(dropped_pages_csv, index=False, encoding="utf-8-sig")
dropped_lines_df.to_csv(dropped_lines_csv, index=False, encoding="utf-8-sig")
source_df.to_csv(source_selection_csv, index=False, encoding="utf-8-sig")

# Per-PDF final text from final_paragraphs, with page separators.
# Sentence-only CSV is intentionally strict and may drop non-terminal fragments.
# The readable final text should preserve clean headings/body paragraphs, so export from paragraphs.
text_export_df = paras_df if not paras_df.empty else sents_df
if not text_export_df.empty:
    for work_id, g in text_export_df.groupby("work_id", dropna=False):
        parts = []
        cur_page = None
        for _, r in g.iterrows():
            if r.get("page_number") != cur_page:
                cur_page = r.get("page_number")
                parts.append(f"\n\n[Page {cur_page}]\n")
            txt = norm_text(r.get("text", ""))
            if txt:
                parts.append(txt)
        (TEXT_DIR / f"{work_id}_final.txt").write_text("\n".join(x for x in parts if norm_text(x)), encoding="utf-8")

# Summary by work_id.
if not page_df.empty:
    pages_summary = page_df.groupby("work_id", dropna=False).agg(
        total_pages=("page_number", "count"),
        kept_pages=("kept_lines", lambda x: int(pd.to_numeric(x, errors="coerce").fillna(0).gt(0).sum())),
        dropped_pages=("selected_source", lambda x: int(x.astype(str).str.contains("dropped|drop", regex=True).sum())),
        avg_selected_quality=("selected_quality_score", "mean") if "selected_quality_score" in page_df.columns else ("class_text_quality_score", "mean"),
        final_sentences=("final_sentences", "sum"),
        dropped_lines=("dropped_lines", "sum"),
    ).reset_index()
else:
    pages_summary = pd.DataFrame()

if not sents_df.empty:
    sent_summary = sents_df.groupby("work_id", dropna=False).agg(
        final_sentence_rows=("sent_id", "count"),
        suspicious_sentence_rows=("suspicious_reasons", lambda x: int(x.astype(str).str.len().gt(0).sum())),
        avg_sentence_quality=("quality_score", "mean"),
        low_vi_ratio_sentences=("vietnamese_ratio", lambda x: int(pd.to_numeric(x, errors="coerce").fillna(0).lt(MIN_VIET_RATIO_FOR_LONG_TEXT).sum())),
    ).reset_index()
    if not pages_summary.empty:
        summary_by_work = pages_summary.merge(sent_summary, on="work_id", how="outer")
    else:
        summary_by_work = sent_summary
else:
    summary_by_work = pages_summary

if not summary_by_work.empty:
    # A rough 0-100 score useful for ranking audit priority, not a guarantee of correctness.
    summary_by_work["quality_score_by_work_id"] = (
        pd.to_numeric(summary_by_work.get("avg_sentence_quality", 0), errors="coerce").fillna(0).clip(0, 100) * 0.55 +
        pd.to_numeric(summary_by_work.get("avg_selected_quality", 0), errors="coerce").fillna(0).clip(0, 100) * 0.45
    ).round(2)
summary_by_work.to_csv(summary_by_work_csv, index=False, encoding="utf-8-sig")

run_summary = pd.DataFrame([{
    "pdf_count": len(pdf_paths),
    "total_pages": int(len(page_df)) if not page_df.empty else processed_pages_total,
    "kept_pages": int(pd.to_numeric(page_df.get("kept_lines", pd.Series(dtype=float)), errors="coerce").fillna(0).gt(0).sum()) if not page_df.empty else 0,
    "dropped_pages": int(len(dropped_pages_df)),
    "total_lines": int(len(lines_df) + len(dropped_lines_df)),
    "final_lines": int(len(lines_df)),
    "dropped_lines": int(len(dropped_lines_df)),
    "final_paragraphs": int(len(paras_df)),
    "final_sentences": int(len(sents_df)),
    "suspicious_lines": int(len(susp_lines_df)),
    "suspicious_sentences": int(len(susp_sents_df)),
    "correction_events": int(len(correction_df)),
    "tesseract_available": tesseract_available(),
    "fast_test_mode": FAST_TEST_MODE,
    "output_dir": str(OUTPUT_DIR),
}])
run_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")

# Package output.
zip_path = PKG_DIR / "dntc_auto_output.zip"
if ZIP_OUTPUT:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in [
            final_lines_csv, final_paras_csv, final_sents_csv, final_sents_auto_csv,
            page_report_csv, line_filter_csv, correction_csv, susp_lines_csv, susp_sents_csv,
            dropped_pages_csv, dropped_lines_csv, source_selection_csv, summary_by_work_csv, summary_csv,
        ]:
            if Path(p).exists():
                zf.write(p, arcname=str(Path(p).relative_to(OUTPUT_DIR)))
        for p in TEXT_DIR.glob("*.txt"):
            zf.write(p, arcname=str(p.relative_to(OUTPUT_DIR)))

print("\n===== RUN SUMMARY =====")
print(run_summary.to_string(index=False))
if not summary_by_work.empty:
    print("\n===== QUALITY BY WORK_ID =====")
    display(summary_by_work.sort_values("quality_score_by_work_id", ascending=True).head(30))
print("\nFinal lines:", final_lines_csv)
print("Final paragraphs:", final_paras_csv)
print("Final sentences only:", final_sents_csv)
print("Compatibility final_sentences_auto:", final_sents_auto_csv)
print("Audit dir:", AUDIT_DIR)
print("output_zip_path:", zip_path if ZIP_OUTPUT else "ZIP_OUTPUT=False")



===== RUN SUMMARY =====
 pdf_count  total_pages  kept_pages  dropped_pages  total_lines  final_lines  dropped_lines  final_paragraphs  final_sentences  suspicious_lines  suspicious_sentences  correction_events  tesseract_available  fast_test_mode                output_dir
        11         2643        2387            255        73158        57835          15323              9565            17630             54510                    52              25596                 True           False /kaggle/working/dntc_auto

===== QUALITY BY WORK_ID =====


,work_id,total_pages,kept_pages,dropped_pages,avg_selected_quality,final_sentences,dropped_lines,final_sentence_rows,suspicious_sentence_rows,avg_sentence_quality,low_vi_ratio_sentences,quality_score_by_work_id
3,09,148,130,18,77.291892,1497,950,1570,3,76.536271,0,76.88
5,12,218,141,76,74.000775,542,1326,558,0,83.701050,0,79.34
8,16_17,321,294,27,79.442922,2508,5141,2522,1,81.571406,0,80.61
9,q2_3_4,502,497,5,80.531952,4129,1319,4260,1,80.897662,0,80.73
6,13,123,108,15,81.757139,594,363,608,1,81.837577,1,81.80
0,01,136,121,15,79.626446,813,1922,818,0,84.504985,0,82.31
4,10_11,133,103,30,83.195350,644,377,664,2,81.963922,2,82.52
2,07_08,220,195,25,81.213046,1296,1736,1328,4,83.718278,0,82.59
7,14_15,173,161,12,83.033578,926,475,943,1,82.745685,1,82.88
1,05,141,112,29,82.667402,660,458,674,0,83.270323,1,83.00



Final lines: /kaggle/working/dntc_auto/final/final_lines.csv
Final paragraphs: /kaggle/working/dntc_auto/final/final_paragraphs.csv
Final sentences only: /kaggle/working/dntc_auto/final/final_sentences_only.csv
Compatibility final_sentences_auto: /kaggle/working/dntc_auto/final/final_sentences_auto.csv
Audit dir: /kaggle/working/dntc_auto/audit
output_zip_path: /kaggle/working/dntc_auto/packages/dntc_auto_output.zip


In [17]:

# ============================================================
# 11. Acceptance sanity checks and examples
# ============================================================
BAD_EXAMPLE_PATTERNS = [
    "OPOCerererore", "Fel Fat Sek", "Seer tit", "mADS ee", ": wa", ". ‘ y", "PHÀM",
    "Đầu thé ky", "doi #ự Đức", "Nha Tuy", "Nước tả", "chi€m", "bi€n", "huyénhién", "Cao Mén",
]

if not sents_df.empty:
    print("final_sentences_only rows:", len(sents_df))
    # final_sentences_only must not contain paragraph/heading rows. It has no type column by design; paragraph_type can be body/footnote.
    print("paragraph_type distribution in sentence-only final:")
    print(sents_df.get("paragraph_type", pd.Series(dtype=str)).value_counts(dropna=False).to_string())
    print("\nBad pattern search in final_sentences_only:")
    for pat in BAD_EXAMPLE_PATTERNS:
        cnt = int(sents_df["text"].astype(str).str.contains(re.escape(pat), case=False, regex=True, na=False).sum())
        print(f"{pat}: {cnt}")
    print("\nTop suspicious sentences audit examples:")
    if not susp_sents_df.empty:
        display(susp_sents_df[[c for c in ["work_id", "page_number", "quality_score", "suspicious_reasons", "text"] if c in susp_sents_df.columns]].head(30))
    else:
        print("No suspicious sentences by current heuristic.")
else:
    print("No final sentences produced. Check dropped_pages.csv and source_selection_report.csv.")

if not dropped_pages_df.empty:
    print("\nDropped pages by class:")
    print(dropped_pages_df.get("page_class", pd.Series(dtype=str)).value_counts(dropna=False).to_string())

if not line_audit_df.empty:
    print("\nLine filter kept/drop counts:")
    print(line_audit_df["kept"].value_counts(dropna=False).to_string())


final_sentences_only rows: 17630
paragraph_type distribution in sentence-only final:
paragraph_type
body        17345
footnote      285

Bad pattern search in final_sentences_only:
OPOCerererore: 0
Fel Fat Sek: 0
Seer tit: 0
mADS ee: 0
: wa: 0
. ‘ y: 0
PHÀM: 64
Đầu thé ky: 0
doi #ự Đức: 0
Nha Tuy: 0
Nước tả: 1
chi€m: 0
bi€n: 0
huyénhién: 0
Cao Mén: 0

Top suspicious sentences audit examples:


,work_id,page_number,quality_score,suspicious_reasons,text
0,05,93,35.398,low_quality:35.4,Lai 1 sé dai 10m80.
1,07_08,47,79.386,weird_char_ratio,Tự điền = điền lệtế-tự.
2,07_08,100,31.600,low_quality:31.6,Cộng lãnh 3 huyện.
3,07_08,112,83.189,library_noise,(thuế ruộng đất) Tự-Đức niên gian cảđiền thổ48...
4,07_08,179,33.967,low_quality:34.0,Người Lệ-Thủy.
5,07_08,203,75.840,weird_char_ratio,黃蠟-Hoàng-Lạp = Sáp ong. 蜂蜜-Phong-Mật = Mật ong...
6,07_08,206,56.392,weird_char_ratio,"Qua = Dưa, bí. 菜 — Quả Thề = Rau."
7,09,78,52.386,weird_char_ratio,Hắc-lăng 黑綾 = lãnh đen.
8,09,78,82.283,weird_char_ratio,Hắc-đường 黑糖 = đường đen.
9,09,78,58.117,weird_char_ratio,Phụng-du 鳳油 = dầu Phụng.



Dropped pages by class:
page_class
blank_page                         91
junk_ocr_page                      62
front_matter_before_content        44
patterned_endpaper                 23
publisher_backmatter_page          22
library_barcode_stamp_watermark     8
cover_title_front_matter            5

Line filter kept/drop counts:
kept
True     57835
False    15323
